# SecureByDesign — Complete Kaggle Notebook
### AI-Powered STRIDE Threat Inference from Data Flow Diagrams
---
**Person B's unified notebook.**  
Run cells top-to-bottom.  
> **Before running:** Add your `GROQ_API_KEY` in *Add-ons → Secrets*.  
> Get a free Groq key at [console.groq.com/keys](https://console.groq.com/keys).

## Phase 0 — Install Dependencies

In [ ]:
!pip install groq>=0.11.0 scikit-learn pandas numpy python-dateutil streamlit pyngrok --quiet

import sklearn, pandas, groq
print(f"✅ groq {groq.__version__}  |  sklearn {sklearn.__version__}  |  pandas {pandas.__version__}")

## Phase 1 — Create Directory Structure & Contract

In [ ]:
import os, json

WORK = "/kaggle/working/SecureByDesign"
dirs = [
    f"{WORK}/pipeline",
    f"{WORK}/evaluation/test_dfds",
    f"{WORK}/evaluation/results",
    f"{WORK}/app",
    f"{WORK}/data",
]
for d in dirs:
    os.makedirs(d, exist_ok=True)
print("✅ Directories created:", os.listdir(WORK))

In [ ]:
import json
contract = {
    "description": "SecureByDesign Data Contract v1.0 — DO NOT MODIFY WITHOUT TEAM AGREEMENT",
    "version": "1.0",
    "input_schema": {
        "dfd_id": "string", "system_name": "string",
        "nodes": [{"id":"string","type":"external_entity|process|datastore","name":"string","description":"string|null"}],
        "edges": [{"id":"string","from":"string","to":"string","data_description":"string|null",
                   "protocol":"string|null","authenticated":"boolean|null","encrypted":"boolean|null"}],
        "trust_boundaries": [{"id":"string","name":"string","separates":["node_id"]}],
        "partial_info_flags": {"missing_trust_boundaries":"boolean","unknown_protocols":"boolean",
                               "unspecified_auth":"boolean","incomplete_nodes":"boolean"}
    },
    "output_schema": {
        "dfd_id":"string","system_name":"string","analysis_timestamp":"ISO8601",
        "overall_risk_level":"Critical|High|Medium|Low","partial_dfd_detected":"boolean",
        "completeness_score":"float 0-1",
        "threats":[{"threat_id":"string","stride_category":"exact STRIDE","affected_component":"string",
                    "threat_description":"string","missing_control":"string","confidence":"High|Medium|Low",
                    "confidence_reason":"string","explanation":"string"}],
        "missing_controls_summary":["string"],
        "stride_coverage":{"Spoofing":"int","Tampering":"int","Repudiation":"int",
                           "Information Disclosure":"int","Denial of Service":"int","Elevation of Privilege":"int"}
    }
}
with open(f"{WORK}/contract.json","w") as f:
    json.dump(contract, f, indent=2)
print("✅ contract.json written")

## Phase 2A — Write Person A's Pipeline to Disk
These cells write the AI inference engine files into `/kaggle/working/SecureByDesign/pipeline/`.

In [ ]:
%%writefile /kaggle/working/SecureByDesign/pipeline/__init__.py
"""
SecureByDesign Pipeline Module
Public API surface — Person B imports from here.

Usage:
    from pipeline.inference import analyze_dfd
    result = analyze_dfd(dfd_json_dict, security_context_string)
"""
from pipeline.inference import analyze_dfd

__all__ = ['analyze_dfd']

In [ ]:
%%writefile /kaggle/working/SecureByDesign/pipeline/dfd_parser.py
"""
DFD Parser for SecureByDesign
Parses DFD JSON into structured context objects for the LLM inference pipeline.

Supports ANY DFD JSON format by dynamically normalizing keys before processing.
Users can submit DFDs with 'name' or 'label', 'from'/'source'/'src',
'datastore'/'data_store'/'database', etc. — all handled automatically.

Author: Person A
Project: SecureByDesign — Explainable LLM-Based STRIDE Threat Inference
"""

import json
import re
from typing import Dict, List, Optional, Tuple, Any
from dataclasses import dataclass, field
from datetime import datetime


# ============================================================
# DYNAMIC KEY ALIAS MAPS
# ============================================================
# Each canonical key → list of alternative keys that should map to it.
# The normalizer walks these in order and uses the first match found.

# Top-level DFD fields
_TOP_LEVEL_ALIASES = {
    'dfd_id':            ['dfd_id', 'id', 'diagram_id', 'dfdId', 'dfd_identifier', 'diagramId'],
    'system_name':       ['system_name', 'name', 'systemName', 'title', 'system', 'project_name',
                          'project', 'application_name', 'app_name', 'service_name'],
    'nodes':             ['nodes', 'elements', 'components', 'entities', 'objects', 'items'],
    'edges':             ['edges', 'flows', 'data_flows', 'dataFlows', 'connections', 'links',
                          'arrows', 'transitions', 'communications', 'interactions'],
    'trust_boundaries':  ['trust_boundaries', 'trustBoundaries', 'boundaries', 'trust_zones',
                          'trustZones', 'zones', 'security_boundaries'],
    'partial_info_flags':['partial_info_flags', 'partialInfoFlags', 'partial_flags', 'flags',
                          'metadata', 'meta'],
}

# Node-level fields
_NODE_ALIASES = {
    'id':              ['id', 'node_id', 'nodeId', 'identifier', 'key', 'uid', 'element_id'],
    'name':            ['name', 'label', 'title', 'display_name', 'displayName', 'node_name',
                        'nodeName', 'text', 'caption', 'description_short'],
    'type':            ['type', 'node_type', 'nodeType', 'kind', 'category', 'element_type',
                        'elementType', 'class', 'role'],
    'description':     ['description', 'desc', 'details', 'notes', 'info', 'summary', 'tooltip'],
    'vulnerabilities': ['vulnerabilities', 'vulns', 'weaknesses', 'issues', 'risks',
                        'security_issues', 'threats', 'findings', 'problems', 'concerns'],
}

# Edge-level fields
_EDGE_ALIASES = {
    'id':               ['id', 'edge_id', 'edgeId', 'flow_id', 'flowId', 'identifier', 'key'],
    'from':             ['from', 'source', 'src', 'from_node', 'fromNode', 'source_id',
                         'sourceId', 'from_id', 'fromId', 'origin', 'start', 'sender'],
    'to':               ['to', 'target', 'dst', 'dest', 'destination', 'to_node', 'toNode',
                         'target_id', 'targetId', 'to_id', 'toId', 'end', 'receiver', 'sink'],
    'label':            ['label', 'name', 'title', 'flow_name', 'flowName', 'display_name',
                         'text', 'caption'],
    'data_description': ['data_description', 'dataDescription', 'data', 'description',
                         'desc', 'details', 'payload', 'content', 'message'],
    'protocol':         ['protocol', 'transport', 'communication_type', 'channel', 'method',
                         'transport_protocol'],
    'authenticated':    ['authenticated', 'auth', 'is_authenticated', 'isAuthenticated',
                         'requires_auth', 'authentication'],
    'encrypted':        ['encrypted', 'is_encrypted', 'isEncrypted', 'encryption', 'tls',
                         'ssl', 'https'],
    'vulnerabilities':  ['vulnerabilities', 'vulns', 'weaknesses', 'issues', 'risks',
                         'security_issues', 'findings'],
}

# Node type normalization: maps various type strings → canonical types
_NODE_TYPE_ALIASES = {
    'external_entity': [
        'external_entity', 'external entity', 'externalentity', 'entity', 'actor',
        'user', 'client', 'external', 'ext_entity', 'external_actor', 'consumer',
        'producer', 'third_party', 'third party', 'api_consumer', 'terminal',
        'interactor', 'external system', 'external_system',
    ],
    'process': [
        'process', 'service', 'function', 'module', 'handler', 'controller',
        'processor', 'logic', 'computation', 'transform', 'microservice',
        'api', 'endpoint', 'server', 'application', 'component', 'task',
        'action', 'operation', 'workflow', 'step', 'activity',
    ],
    'datastore': [
        'datastore', 'data_store', 'data store', 'database', 'db', 'storage',
        'repository', 'cache', 'file', 'filesystem', 'file_system', 'queue',
        'message_queue', 'data', 'table', 'collection', 'bucket', 'store',
        'persistence', 'log', 'log_store', 'data_lake', 'warehouse',
    ],
}


# ============================================================
# NORMALIZER
# ============================================================

def _resolve_key(source: dict, aliases: List[str], default=None):
    """Return the value of the first matching alias key found in source."""
    for alias in aliases:
        if alias in source:
            return source[alias]
    # Try case-insensitive match as last resort
    lower_map = {k.lower().replace('-', '_').replace(' ', '_'): k for k in source}
    for alias in aliases:
        normalized = alias.lower().replace('-', '_').replace(' ', '_')
        if normalized in lower_map:
            return source[lower_map[normalized]]
    return default


def _normalize_node_type(raw_type: str) -> str:
    """Map any node type string to one of the three canonical types."""
    cleaned = raw_type.lower().strip().replace('-', '_').replace(' ', '_')
    for canonical, variants in _NODE_TYPE_ALIASES.items():
        if cleaned in [v.replace(' ', '_') for v in variants]:
            return canonical
    # Fallback: check if the type string *contains* a known keyword
    for canonical, variants in _NODE_TYPE_ALIASES.items():
        for v in variants:
            if v.replace(' ', '_') in cleaned or cleaned in v.replace(' ', '_'):
                return canonical
    return raw_type  # Return as-is; parser will treat as process


def _normalize_node(raw_node: dict) -> dict:
    """Normalize a single node dict to canonical keys."""
    normalized = {}

    # Preserve ALL original keys (so extra data like vulnerabilities are kept)
    for k, v in raw_node.items():
        normalized[k] = v

    # Override with canonical keys
    for canonical_key, aliases in _NODE_ALIASES.items():
        val = _resolve_key(raw_node, aliases)
        if val is not None:
            normalized[canonical_key] = val

    # Normalize the type field
    if 'type' in normalized:
        normalized['type'] = _normalize_node_type(str(normalized['type']))

    # Ensure 'name' field exists (fallback chain: name → label → id → 'Unknown')
    if 'name' not in normalized or not normalized['name']:
        normalized['name'] = normalized.get('label', normalized.get('id', 'Unknown'))

    return normalized


def _normalize_edge(raw_edge: dict) -> dict:
    """Normalize a single edge dict to canonical keys."""
    normalized = {}

    # Preserve ALL original keys
    for k, v in raw_edge.items():
        normalized[k] = v

    # Override with canonical keys
    for canonical_key, aliases in _EDGE_ALIASES.items():
        val = _resolve_key(raw_edge, aliases)
        if val is not None:
            normalized[canonical_key] = val

    # If no data_description but a label exists, use label as data description
    if 'data_description' not in normalized and 'label' in normalized:
        normalized['data_description'] = normalized['label']

    return normalized


def normalize_dfd_json(dfd_json: dict) -> dict:
    """
    DYNAMIC DFD NORMALIZER
    ======================
    Accepts ANY DFD JSON format and normalizes it to the canonical schema
    expected by the parser.

    Handles:
    - Alternative key names (label/name, source/from, target/to, etc.)
    - Alternative node type values (entity/actor/user → external_entity, etc.)
    - CamelCase / snake_case / mixed-case keys
    - Alternative top-level field names (components/elements → nodes, etc.)
    - Missing fields (provides safe defaults)
    - Preserves all extra/unknown fields for downstream use

    This function should be called BEFORE parse_dfd() to guarantee
    format-agnostic parsing.
    """
    normalized = {}

    # ── Resolve top-level fields ──────────────────────────────────────────────
    for canonical_key, aliases in _TOP_LEVEL_ALIASES.items():
        val = _resolve_key(dfd_json, aliases)
        if val is not None:
            normalized[canonical_key] = val

    # ── Defaults for required top-level fields ────────────────────────────────
    if 'dfd_id' not in normalized:
        # Auto-generate a DFD ID if missing
        normalized['dfd_id'] = f"auto_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

    if 'system_name' not in normalized:
        normalized['system_name'] = 'Unknown System'

    if 'nodes' not in normalized:
        normalized['nodes'] = []

    if 'edges' not in normalized:
        normalized['edges'] = []

    # ── Normalize each node ───────────────────────────────────────────────────
    raw_nodes = normalized.get('nodes', [])
    normalized_nodes = []
    for i, node in enumerate(raw_nodes):
        if isinstance(node, dict):
            n = _normalize_node(node)
            # Ensure every node has an id
            if 'id' not in n:
                n['id'] = f"node_{i}"
            normalized_nodes.append(n)
        elif isinstance(node, str):
            # Handle case where nodes are just strings (names)
            normalized_nodes.append({
                'id': f"node_{i}",
                'name': node,
                'type': 'process',
            })
    normalized['nodes'] = normalized_nodes

    # ── Normalize each edge ───────────────────────────────────────────────────
    raw_edges = normalized.get('edges', [])
    normalized_edges = []
    for i, edge in enumerate(raw_edges):
        if isinstance(edge, dict):
            e = _normalize_edge(edge)
            # Ensure every edge has an id
            if 'id' not in e:
                e['id'] = f"edge_{i}"
            normalized_edges.append(e)
    normalized['edges'] = normalized_edges

    # ── Normalize trust boundaries (if present) ───────────────────────────────
    raw_tb = normalized.get('trust_boundaries', [])
    normalized_tb = []
    for tb in raw_tb:
        if isinstance(tb, dict):
            norm_tb = dict(tb)
            # Resolve name
            if 'name' not in norm_tb:
                norm_tb['name'] = norm_tb.get('label', norm_tb.get('title',
                                  norm_tb.get('id', 'Unnamed Boundary')))
            # Resolve separates
            if 'separates' not in norm_tb:
                norm_tb['separates'] = norm_tb.get('contains', norm_tb.get(
                    'members', norm_tb.get('nodes', [])))
            normalized_tb.append(norm_tb)
    normalized['trust_boundaries'] = normalized_tb

    # ── Preserve any extra top-level keys not in alias map ────────────────────
    known_keys = set()
    for aliases in _TOP_LEVEL_ALIASES.values():
        known_keys.update(aliases)
    for k, v in dfd_json.items():
        if k not in normalized and k not in known_keys:
            normalized[k] = v

    return normalized


# ============================================================
# DATA STRUCTURES
# ============================================================

@dataclass
class ParsedDFD:
    """
    Structured representation of a parsed DFD, ready for prompt construction.
    All fields are populated by parse_dfd() — never construct directly.
    """
    dfd_id: str
    system_name: str

    # Node groups (categorized by type)
    external_entities: List[Dict] = field(default_factory=list)
    processes: List[Dict] = field(default_factory=list)
    datastores: List[Dict] = field(default_factory=list)

    # Edges and boundaries
    edges: List[Dict] = field(default_factory=list)
    trust_boundaries: List[Dict] = field(default_factory=list)

    # Security-relevant analysis flags
    boundary_crossing_edges: List[Dict] = field(default_factory=list)  # Edges crossing trust boundaries
    missing_auth_edges: List[str] = field(default_factory=list)        # Edge IDs with null/unspecified auth
    missing_encryption_edges: List[str] = field(default_factory=list)  # Edge IDs with null/unspecified encryption
    unknown_protocol_edges: List[str] = field(default_factory=list)    # Edge IDs with null/unknown protocol

    # Partial DFD detection
    is_partial: bool = False
    completeness_score: float = 1.0  # 0.0 = almost no info, 1.0 = fully specified
    missing_elements: List[str] = field(default_factory=list)

    # LLM-ready text summary (built by _build_text_summary)
    text_summary: str = ""


# Convenience property — all nodes regardless of type
ParsedDFD.nodes_all = property(
    lambda self: self.external_entities + self.processes + self.datastores
)


# ============================================================
# MAIN PARSING FUNCTION
# ============================================================

def parse_dfd(dfd_json: dict) -> ParsedDFD:
    """
    Main entry point. Parse a DFD JSON dict into a ParsedDFD object.

    THIS FUNCTION IS FORMAT-AGNOSTIC. It accepts any DFD JSON structure
    and automatically normalizes it before parsing.

    Performs:
      - Dynamic key normalization (handles name/label, from/source, etc.)
      - Schema validation (required fields)
      - Node categorization by type (with fuzzy type matching)
      - Edge security property analysis
      - Trust boundary crossing detection
      - Completeness scoring and partial DFD detection
      - LLM-ready text summary generation

    Args:
        dfd_json: Raw DFD dictionary in ANY common format.
                  Will be auto-normalized to canonical schema.

    Returns:
        ParsedDFD object with all fields populated.

    Raises:
        ValueError: If the DFD JSON is completely unparseable (no nodes at all).
    """

    # ── STEP 0: Normalize to canonical format ─────────────────────────────────
    dfd_json = normalize_dfd_json(dfd_json)

    # ── Validate required fields (now guaranteed by normalizer) ───────────────
    required_fields = ['dfd_id', 'system_name', 'nodes', 'edges']
    for f in required_fields:
        if f not in dfd_json:
            raise ValueError(
                f"Missing required field in DFD JSON: '{f}'. "
                f"Required fields: {required_fields}"
            )

    parsed = ParsedDFD(
        dfd_id=dfd_json['dfd_id'],
        system_name=dfd_json['system_name']
    )

    # Build node lookup dict for name resolution in edges/boundaries
    node_lookup: Dict[str, Dict] = {
        node['id']: node for node in dfd_json.get('nodes', [])
        if 'id' in node
    }

    # ── STEP 1: Categorize nodes by type ─────────────────────────────────────
    for node in dfd_json.get('nodes', []):
        node_type = node.get('type', '').lower().strip()

        if node_type == 'external_entity':
            parsed.external_entities.append(node)
        elif node_type == 'process':
            parsed.processes.append(node)
        elif node_type == 'datastore':
            parsed.datastores.append(node)
        else:
            # Unknown node type — treat as process for analysis purposes
            # Do not drop; unknown types in real-world DFDs are common
            node_copy = dict(node)
            node_copy['_type_note'] = f"Unknown type '{node_type}' — treated as process"
            parsed.processes.append(node_copy)

    # ── STEP 2: Process edges — detect missing security properties ─────────────
    parsed.edges = dfd_json.get('edges', [])

    for edge in parsed.edges:
        edge_id = edge.get('id', 'unknown')

        # null = unspecified (our schema uses None for unknowns)
        if edge.get('authenticated') is None:
            parsed.missing_auth_edges.append(edge_id)

        if edge.get('encrypted') is None:
            parsed.missing_encryption_edges.append(edge_id)

        if edge.get('protocol') is None:
            parsed.unknown_protocol_edges.append(edge_id)

    # ── STEP 3: Load trust boundaries ────────────────────────────────────────
    parsed.trust_boundaries = dfd_json.get('trust_boundaries', [])

    # ── STEP 4: Detect trust boundary crossings ───────────────────────────────
    # An edge crosses a boundary if exactly one of its endpoints appears in
    # the boundary's 'separates' list. This is the key attack surface signal.
    for edge in parsed.edges:
        from_node = edge.get('from')
        to_node = edge.get('to')

        for boundary in parsed.trust_boundaries:
            separated = set(boundary.get('separates', []))

            # XOR: one endpoint inside, one outside
            from_inside = from_node in separated
            to_inside = to_node in separated

            if from_inside != to_inside:
                crossing_info = {
                    **edge,
                    'crossing_boundary': boundary.get('id', 'unknown'),
                    'boundary_name': boundary.get('name', 'Unknown Boundary'),
                    'from_name': node_lookup.get(from_node, {}).get('name', from_node),
                    'to_name': node_lookup.get(to_node, {}).get('name', to_node),
                }
                parsed.boundary_crossing_edges.append(crossing_info)
                break  # Each edge crosses at most one boundary in simplified model

    # ── STEP 5: Compute completeness score & detect partial DFD ──────────────
    partial_flags = dfd_json.get('partial_info_flags', {})
    if not isinstance(partial_flags, dict):
        partial_flags = {}

    missing_elements: List[str] = []
    completeness_factors: List[float] = []

    # Factor 1: Trust boundaries present?
    has_tb = bool(parsed.trust_boundaries)
    tb_flagged_missing = partial_flags.get('missing_trust_boundaries', False)
    if not has_tb or tb_flagged_missing:
        missing_elements.append(
            "Trust boundaries not defined — cannot determine attack surface perimeter"
        )
        completeness_factors.append(0.0)
    else:
        completeness_factors.append(1.0)

    # Factor 2: Authentication specified on edges
    total_edges = max(len(parsed.edges), 1)
    auth_ratio = 1.0 - (len(parsed.missing_auth_edges) / total_edges)
    completeness_factors.append(auth_ratio)
    if parsed.missing_auth_edges:
        missing_elements.append(
            f"Authentication unspecified on edges: {', '.join(parsed.missing_auth_edges)}"
        )

    # Factor 3: Encryption specified on edges
    enc_ratio = 1.0 - (len(parsed.missing_encryption_edges) / total_edges)
    completeness_factors.append(enc_ratio)
    if parsed.missing_encryption_edges:
        missing_elements.append(
            f"Encryption unspecified on edges: {', '.join(parsed.missing_encryption_edges)}"
        )

    # Factor 4: Protocols specified on edges
    proto_flagged = partial_flags.get('unknown_protocols', False)
    if proto_flagged or parsed.unknown_protocol_edges:
        missing_elements.append(
            f"Protocol unknown on edges: {', '.join(parsed.unknown_protocol_edges) or 'flagged in partial_info_flags'}"
        )
        completeness_factors.append(0.5)
    else:
        completeness_factors.append(1.0)

    # Guard: empty DFD
    if len(parsed.nodes_all) == 0:
        missing_elements.append("No nodes defined — DFD is empty")
        completeness_factors.append(0.0)

    if len(parsed.edges) == 0:
        missing_elements.append("No data flows defined — cannot analyze data movement")
        completeness_factors.append(0.0)

    parsed.completeness_score = sum(completeness_factors) / max(len(completeness_factors), 1)
    parsed.is_partial = parsed.completeness_score < 0.8 or bool(missing_elements)
    parsed.missing_elements = missing_elements

    # ── STEP 6: Build LLM-ready text summary ─────────────────────────────────
    parsed.text_summary = _build_text_summary(parsed, node_lookup)

    return parsed


# ============================================================
# TEXT SUMMARY BUILDER
# ============================================================

def _build_text_summary(parsed: ParsedDFD, node_lookup: Dict[str, Dict]) -> str:
    """
    Build a structured natural language summary of the DFD for use in LLM prompts.
    Designed to be maximally informative for STRIDE threat reasoning.
    Highlights trust boundary crossings and missing security properties prominently.

    Returns:
        Multi-line string ready for inclusion in an LLM prompt.
    """
    lines = []

    # Header
    lines.append(f"SYSTEM: {parsed.system_name}")
    lines.append(f"DFD ID: {parsed.dfd_id}")
    lines.append(f"COMPLETENESS: {parsed.completeness_score:.0%}")
    lines.append("")

    # Components
    lines.append("=== COMPONENTS ===")
    if parsed.external_entities:
        names = ', '.join(n.get('name', n.get('id', 'Unknown')) for n in parsed.external_entities)
        lines.append(f"External Entities (untrusted/outside system boundary): {names}")

        # Include vulnerabilities if present
        for n in parsed.external_entities:
            vulns = n.get('vulnerabilities', [])
            if vulns:
                lines.append(f"  ⚠ {n.get('name', n.get('id'))}: {'; '.join(vulns)}")

    if parsed.processes:
        names = ', '.join(n.get('name', n.get('id', 'Unknown')) for n in parsed.processes)
        lines.append(f"Processes (application logic): {names}")

        for n in parsed.processes:
            vulns = n.get('vulnerabilities', [])
            if vulns:
                lines.append(f"  ⚠ {n.get('name', n.get('id'))}: {'; '.join(vulns)}")

    if parsed.datastores:
        names = ', '.join(n.get('name', n.get('id', 'Unknown')) for n in parsed.datastores)
        lines.append(f"Datastores (data at rest): {names}")

        for n in parsed.datastores:
            vulns = n.get('vulnerabilities', [])
            if vulns:
                lines.append(f"  ⚠ {n.get('name', n.get('id'))}: {'; '.join(vulns)}")

    if not parsed.nodes_all:
        lines.append("  WARNING: No components defined. DFD is empty.")
    lines.append("")

    # Trust boundaries
    lines.append("=== TRUST BOUNDARIES ===")
    if parsed.trust_boundaries:
        for tb in parsed.trust_boundaries:
            separated_ids = tb.get('separates', [])
            separated_names = [node_lookup.get(n, {}).get('name', n) for n in separated_ids]
            lines.append(f"  [{tb.get('id', '?')}] {tb.get('name', 'Unnamed Boundary')}: "
                         f"separates {' | '.join(separated_names)}")
    else:
        lines.append("  WARNING: No trust boundaries defined. Attack surface cannot be determined.")
    lines.append("")

    # Data flows (all edges)
    lines.append("=== DATA FLOWS ===")
    if parsed.edges:
        for edge in parsed.edges:
            from_name = node_lookup.get(edge.get('from'), {}).get('name', edge.get('from', '?'))
            to_name = node_lookup.get(edge.get('to'), {}).get('name', edge.get('to', '?'))

            # Edge label (if present)
            edge_label = edge.get('label', edge.get('data_description', ''))

            # Authentication status
            auth_val = edge.get('authenticated')
            if auth_val is True:
                auth_str = "authenticated"
            elif auth_val is False:
                auth_str = "NOT authenticated"
            else:
                auth_str = "AUTHENTICATION UNSPECIFIED"

            # Encryption status
            enc_val = edge.get('encrypted')
            if enc_val is True:
                enc_str = "encrypted"
            elif enc_val is False:
                enc_str = "NOT encrypted"
            else:
                enc_str = "ENCRYPTION UNSPECIFIED"

            proto = edge.get('protocol') or "UNKNOWN PROTOCOL"
            data = edge.get('data_description') or edge_label or "unspecified data"

            lines.append(f"  [{edge.get('id', '?')}] {from_name} → {to_name}")
            lines.append(f"       Data: {data} | Protocol: {proto} | {auth_str} | {enc_str}")

            # Include edge vulnerabilities if present
            vulns = edge.get('vulnerabilities', [])
            if vulns:
                lines.append(f"       ⚠ Vulnerabilities: {'; '.join(vulns)}")
    else:
        lines.append("  WARNING: No data flows defined.")
    lines.append("")

    # Trust boundary crossings — highest-priority section for STRIDE analysis
    if parsed.boundary_crossing_edges:
        lines.append("=== TRUST BOUNDARY CROSSINGS (HIGH SECURITY RELEVANCE) ===")
        for crossing in parsed.boundary_crossing_edges:
            lines.append(f"  [{crossing.get('id', '?')}] {crossing.get('from_name', '?')} → {crossing.get('to_name', '?')}")
            lines.append(f"       Crosses: {crossing.get('boundary_name', 'Unknown Boundary')}")
            auth_val = crossing.get('authenticated')
            auth = "UNSPECIFIED" if auth_val is None else ("YES" if auth_val else "NO")
            enc_val = crossing.get('encrypted')
            enc = "UNSPECIFIED" if enc_val is None else ("YES" if enc_val else "NO")
            lines.append(f"       Auth: {auth} | Encrypted: {enc}")
        lines.append("")

    # Partial DFD warnings — give the LLM clear signals about uncertainty
    if parsed.is_partial:
        lines.append("=== PARTIAL DFD WARNINGS — REASON FOR LOWER CONFIDENCE ===")
        for warning in parsed.missing_elements:
            lines.append(f"  ⚠  {warning}")
        lines.append("")

    return "\n".join(lines)


# ============================================================
# SELF-TEST (run directly to verify)
# ============================================================

if __name__ == "__main__":
    print("=" * 60)
    print("DFD PARSER SELF-TEST")
    print("=" * 60)

    # TEST 1: Standard format with 'name' key
    print("\n--- Test 1: Standard 'name' format ---")
    sample_dfd = {
        "dfd_id": "test_001",
        "system_name": "Test Payment Service",
        "nodes": [
            {"id": "N1", "type": "external_entity", "name": "Mobile Client", "description": "User app"},
            {"id": "N2", "type": "process", "name": "API Gateway", "description": "Entry point"},
            {"id": "N3", "type": "datastore", "name": "User DB", "description": "User data"}
        ],
        "edges": [
            {"id": "E1", "from": "N1", "to": "N2", "data_description": "Login request",
             "protocol": "HTTPS", "authenticated": None, "encrypted": True},
            {"id": "E2", "from": "N2", "to": "N3", "data_description": "DB query",
             "protocol": None, "authenticated": None, "encrypted": None}
        ],
        "trust_boundaries": [
            {"id": "TB1", "name": "Internet Boundary", "separates": ["N1", "N2"]}
        ],
        "partial_info_flags": {
            "missing_trust_boundaries": False,
            "unknown_protocols": True,
            "unspecified_auth": True,
            "incomplete_nodes": False
        }
    }

    result = parse_dfd(sample_dfd)
    print(f"✅ System: {result.system_name}")
    print(f"✅ Completeness: {result.completeness_score:.0%}")
    print(f"✅ Total Nodes: {len(result.nodes_all)}")
    assert len(result.nodes_all) == 3, "Expected 3 nodes"

    # TEST 2: 'label' format (like the user's DFD)
    print("\n--- Test 2: 'label' format ---")
    label_dfd = {
        "dfd_id": "DFD-001",
        "system_name": "Online Banking Web Application",
        "nodes": [
            {"id": "E1", "type": "external_entity", "label": "Customer"},
            {"id": "P1", "type": "process", "label": "Login Process",
             "vulnerabilities": ["No rate limiting", "No MFA"]},
            {"id": "D1", "type": "data_store", "label": "User Database",
             "vulnerabilities": ["Passwords in plaintext"]},
        ],
        "edges": [
            {"from": "E1", "to": "P1", "label": "Submit Credentials (HTTP)",
             "vulnerabilities": ["Unencrypted transmission"]},
            {"from": "P1", "to": "D1", "label": "Validate Credentials"},
        ]
    }

    result2 = parse_dfd(label_dfd)
    print(f"✅ System: {result2.system_name}")
    print(f"✅ External Entities: {[n['name'] for n in result2.external_entities]}")
    print(f"✅ Processes: {[n['name'] for n in result2.processes]}")
    print(f"✅ Datastores: {[n['name'] for n in result2.datastores]}")
    assert result2.external_entities[0]['name'] == "Customer"
    assert result2.datastores[0]['name'] == "User Database"

    # TEST 3: source/target + alternative type names
    print("\n--- Test 3: Alternative keys (source/target, actor/database) ---")
    alt_dfd = {
        "id": "ALT-001",
        "title": "E-Commerce Platform",
        "elements": [
            {"id": "A1", "type": "actor", "label": "Buyer"},
            {"id": "S1", "type": "service", "name": "Order Service"},
            {"id": "DB1", "type": "database", "label": "Orders DB"},
        ],
        "flows": [
            {"source": "A1", "target": "S1", "label": "Place Order"},
            {"source": "S1", "target": "DB1", "label": "Save Order"},
        ]
    }

    result3 = parse_dfd(alt_dfd)
    print(f"✅ System: {result3.system_name}")
    print(f"✅ DFD ID: {result3.dfd_id}")
    print(f"✅ External Entities: {[n['name'] for n in result3.external_entities]}")
    print(f"✅ Processes: {[n['name'] for n in result3.processes]}")
    print(f"✅ Datastores: {[n['name'] for n in result3.datastores]}")
    print(f"✅ Edges: {len(result3.edges)}")
    assert result3.system_name == "E-Commerce Platform"
    assert result3.dfd_id == "ALT-001"
    assert len(result3.external_entities) == 1  # actor → external_entity
    assert len(result3.datastores) == 1          # database → datastore
    assert len(result3.edges) == 2               # flows → edges

    # TEST 4: Minimal / edge case
    print("\n--- Test 4: Minimal DFD ---")
    minimal = {
        "dfd_id": "minimal_001",
        "system_name": "Minimal System",
        "nodes": [{"id": "N1", "type": "process", "name": "Service"}],
        "edges": []
    }
    result4 = parse_dfd(minimal)
    print(f"✅ Completeness: {result4.completeness_score:.0%}")
    print(f"✅ Is Partial: {result4.is_partial}")

    print("\n" + "=" * 60)
    print("✅ ALL DFD PARSER TESTS PASSED")
    print("=" * 60)


In [ ]:
%%writefile /kaggle/working/SecureByDesign/pipeline/prompt_templates.py
"""
Prompt Templates for SecureByDesign
All LLM prompts for STRIDE threat inference from DFD JSON.

Author: Person A
Project: SecureByDesign — Explainable LLM-Based STRIDE Threat Inference

ENGINEERING NOTES:
  - System prompt establishes STRIDE expert persona and enforces JSON-only output
  - Three few-shot examples cover: complete DFD, highly incomplete DFD, internal microservice
  - Partial DFD instructions tell the model to LOWER confidence, not refuse to analyze
  - JSON output is always enforced — no markdown, no preamble, no explanation outside JSON
  - Temperature should be set to 0.1 in the model config (done in inference.py)
"""


# ============================================================
# SYSTEM PROMPT
# Sets LLM persona, rules, output schema, and handling of partial DFDs
# ============================================================

SYSTEM_PROMPT = """You are an expert software security architect specializing in threat modeling using the STRIDE framework. You have 15 years of experience analyzing Data Flow Diagrams (DFDs) and identifying architectural security risks in enterprise microservice systems, payment platforms, and cloud-native architectures.

Your task is to analyze a software system's Data Flow Diagram and produce a comprehensive STRIDE threat analysis. You must follow ALL of these rules precisely:

ANALYSIS RULES:
1. Identify all credible security threats based ONLY on the DFD information provided
2. Map each threat to EXACTLY ONE STRIDE category:
   Spoofing | Tampering | Repudiation | Information Disclosure | Denial of Service | Elevation of Privilege
3. Reference specific DFD components (exact node names and edge IDs like E1, E2) in every threat finding
4. Assign confidence based strictly on information completeness:
   - HIGH: Explicit evidence in DFD — confirmed missing control or confirmed insecure configuration
   - MEDIUM: Probable risk based on common patterns — some DFD information is absent
   - LOW: Possible risk but substantial information is missing — reasoning is largely inferential
5. For INCOMPLETE/PARTIAL DFDs: Still produce findings but lower your confidence accordingly — do NOT refuse to analyze or say you cannot determine anything
6. Write architect-facing explanations: practical, specific to this system's components, actionable

CRITICAL OUTPUT RULES:
- Respond ONLY with valid JSON. No markdown. No preamble. No explanation outside the JSON object.
- Your response MUST start exactly with { and end exactly with }
- Every threat object MUST have ALL required fields populated (no nulls, no omissions)
- Do NOT invent threats that have no basis in the DFD — ground every finding in specific edges, nodes, or missing boundaries
- Do NOT claim a system is "secure" — always note what cannot be verified from the available information
- Produce at minimum 2 threats and at maximum 8 threats per analysis

OUTPUT SCHEMA (follow exactly):
{
  "dfd_id": "string — copy from input",
  "system_name": "string — copy from input",
  "analysis_timestamp": "ISO 8601 timestamp (e.g. 2025-01-01T10:00:00Z)",
  "overall_risk_level": "Critical | High | Medium | Low",
  "partial_dfd_detected": true or false,
  "threats": [
    {
      "threat_id": "T1",
      "stride_category": "exact STRIDE category name",
      "affected_component": "e.g. E1: Mobile Client → API Gateway",
      "threat_description": "concise, specific description of what the threat is",
      "missing_control": "the specific security control that is absent",
      "confidence": "High | Medium | Low",
      "confidence_reason": "one sentence: why this specific confidence level was assigned",
      "explanation": "2-3 sentences in plain English for an architect: what happens, why it matters, what to do"
    }
  ],
  "missing_controls_summary": ["array of top-level missing security controls — 2-5 items"],
  "stride_coverage": {
    "Spoofing": 0,
    "Tampering": 0,
    "Repudiation": 0,
    "Information Disclosure": 0,
    "Denial of Service": 0,
    "Elevation of Privilege": 0
  }
}"""


# ============================================================
# FEW-SHOT EXAMPLE 1
# Scenario: E-Commerce Order Service — 75% complete, clear High-confidence threats
# Teaches: High-confidence findings on explicitly missing controls
# ============================================================

FEW_SHOT_EXAMPLE_1_INPUT = """Analyze the following Data Flow Diagram for STRIDE security threats. Respond with ONLY valid JSON matching the output schema. No markdown. No preamble.

SYSTEM: E-Commerce Order Service
DFD ID: example_001
COMPLETENESS: 75%

=== COMPONENTS ===
External Entities (untrusted/outside system boundary): Web Browser
Processes (application logic): Order API, Payment Processor
Datastores (data at rest): Orders Database, Payment Vault

=== TRUST BOUNDARIES ===
  [TB1] Internet Boundary: separates Web Browser | Order API

=== DATA FLOWS ===
  [E1] Web Browser → Order API
       Data: Order request with payment info | Protocol: HTTPS | AUTHENTICATION UNSPECIFIED | encrypted
  [E2] Order API → Payment Processor
       Data: Payment card data | Protocol: HTTP | NOT authenticated | NOT encrypted
  [E3] Payment Processor → Payment Vault
       Data: Tokenized payment data | Protocol: TCP | AUTHENTICATION UNSPECIFIED | ENCRYPTION UNSPECIFIED
  [E4] Order API → Orders Database
       Data: Order records | Protocol: TCP | NOT authenticated | encrypted

=== TRUST BOUNDARY CROSSINGS (HIGH SECURITY RELEVANCE) ===
  [E1] Web Browser → Order API
       Crosses: Internet Boundary
       Auth: UNSPECIFIED | Encrypted: YES

Produce the complete STRIDE threat analysis now:"""

FEW_SHOT_EXAMPLE_1_OUTPUT = """{
  "dfd_id": "example_001",
  "system_name": "E-Commerce Order Service",
  "analysis_timestamp": "2025-01-01T00:00:00Z",
  "overall_risk_level": "Critical",
  "partial_dfd_detected": true,
  "threats": [
    {
      "threat_id": "T1",
      "stride_category": "Information Disclosure",
      "affected_component": "E2: Order API → Payment Processor",
      "threat_description": "Payment card data transmitted over unencrypted HTTP between internal services",
      "missing_control": "Enforce TLS/HTTPS on E2; implement end-to-end encryption for all payment data in transit",
      "confidence": "High",
      "confidence_reason": "Edge E2 is explicitly marked NOT encrypted while carrying payment card data — a confirmed critical violation",
      "explanation": "Edge E2 carries raw payment card data between the Order API and Payment Processor over plain HTTP with no encryption. Any attacker with access to the internal network — such as a malicious insider or a compromised service — can intercept and read card numbers verbatim. This is a PCI-DSS violation. Immediately replace HTTP with mutual TLS on this connection."
    },
    {
      "threat_id": "T2",
      "stride_category": "Spoofing",
      "affected_component": "E1: Web Browser → Order API (crosses Internet Boundary TB1)",
      "threat_description": "No authentication mechanism specified on the only internet-facing entry point",
      "missing_control": "Implement JWT/OAuth 2.0 token validation at the Order API before processing any request",
      "confidence": "High",
      "confidence_reason": "E1 crosses the internet trust boundary with authentication explicitly marked UNSPECIFIED",
      "explanation": "Any request entering the system through E1 is from the untrusted internet. With authentication unspecified, an attacker can submit arbitrary order requests while impersonating any user identity. Implement server-side JWT validation so every request is bound to a verified identity before execution."
    },
    {
      "threat_id": "T3",
      "stride_category": "Tampering",
      "affected_component": "E4: Order API → Orders Database",
      "threat_description": "Unauthenticated database connection allows any process to modify order records",
      "missing_control": "Enforce database-level authentication using least-privilege service accounts with parameterized queries",
      "confidence": "High",
      "confidence_reason": "E4 is explicitly NOT authenticated on a datastore containing financial records",
      "explanation": "The connection from Order API to Orders Database on E4 has no authentication enforced. A compromised process anywhere in the system can freely query, modify, or delete order records without restriction. Implement authenticated service-account access with role-based permissions scoped to exactly what the Order API requires."
    },
    {
      "threat_id": "T4",
      "stride_category": "Repudiation",
      "affected_component": "Order API (process)",
      "threat_description": "No audit logging mechanism present for order or payment processing actions",
      "missing_control": "Implement immutable audit logs for all order creation, modification, cancellation, and payment events",
      "confidence": "Medium",
      "confidence_reason": "Audit logging is not modeled in the DFD — may exist but cannot be confirmed from available information",
      "explanation": "The DFD models no logging or audit trail component for the Order API or Payment Processor. Without tamper-evident logs, a malicious actor can place fraudulent orders and credibly deny it. Implement write-once audit logs capturing actor, action, timestamp, and affected record for every order operation."
    },
    {
      "threat_id": "T5",
      "stride_category": "Denial of Service",
      "affected_component": "E1: Web Browser → Order API (crosses Internet Boundary TB1)",
      "threat_description": "No rate limiting or throttling mechanism on the internet-facing Order API entry point",
      "missing_control": "Implement rate limiting, request throttling, and DDoS protection at the API Gateway layer",
      "confidence": "Medium",
      "confidence_reason": "No rate limiting component modeled in DFD; absence of defense-in-depth for internet-facing entry point",
      "explanation": "The single internet-facing entry point E1 shows no rate limiting or request throttling controls. An attacker could flood the Order API with requests, exhausting backend resources and making the system unavailable to legitimate users. Implement per-IP and per-user rate limits with exponential backoff enforcement."
    }
  ],
  "missing_controls_summary": [
    "Encryption absent on internal payment data flow E2 — Critical PCI-DSS violation requiring immediate remediation",
    "Authentication unspecified on internet-facing entry point E1 — spoofing risk",
    "Database authentication absent on Orders Database connection E4 — tampering risk",
    "No audit logging modeled — repudiation risk on all financial operations"
  ],
  "stride_coverage": {
    "Spoofing": 1,
    "Tampering": 1,
    "Repudiation": 1,
    "Information Disclosure": 1,
    "Denial of Service": 1,
    "Elevation of Privilege": 0
  }
}"""


# ============================================================
# FEW-SHOT EXAMPLE 2
# Scenario: Auth Service — 35% complete, barely any info
# Teaches: How to handle partial DFDs with MEDIUM/LOW confidence
# ============================================================

FEW_SHOT_EXAMPLE_2_INPUT = """Analyze the following Data Flow Diagram for STRIDE security threats. Respond with ONLY valid JSON matching the output schema. No markdown. No preamble.

SYSTEM: Microservice Auth Service
DFD ID: example_002
COMPLETENESS: 35%

=== COMPONENTS ===
External Entities (untrusted/outside system boundary): API Client
Processes (application logic): Auth Service
Datastores (data at rest): Token Store

=== TRUST BOUNDARIES ===
  WARNING: No trust boundaries defined. Attack surface cannot be determined.

=== DATA FLOWS ===
  [E1] API Client → Auth Service
       Data: Credentials | Protocol: UNKNOWN PROTOCOL | AUTHENTICATION UNSPECIFIED | ENCRYPTION UNSPECIFIED
  [E2] Auth Service → Token Store
       Data: Token data | Protocol: UNKNOWN PROTOCOL | AUTHENTICATION UNSPECIFIED | ENCRYPTION UNSPECIFIED

=== PARTIAL DFD WARNINGS — REASON FOR LOWER CONFIDENCE ===
  ⚠  Trust boundaries not defined — cannot determine attack surface perimeter
  ⚠  Authentication unspecified on edges: E1, E2
  ⚠  Encryption unspecified on edges: E1, E2
  ⚠  Protocol unknown on edges: E1, E2

Produce the complete STRIDE threat analysis now:"""

FEW_SHOT_EXAMPLE_2_OUTPUT = """{
  "dfd_id": "example_002",
  "system_name": "Microservice Auth Service",
  "analysis_timestamp": "2025-01-01T00:00:00Z",
  "overall_risk_level": "High",
  "partial_dfd_detected": true,
  "threats": [
    {
      "threat_id": "T1",
      "stride_category": "Spoofing",
      "affected_component": "E1: API Client → Auth Service",
      "threat_description": "Credential flow uses unknown protocol with unspecified authentication — potential identity spoofing",
      "missing_control": "Specify and enforce HTTPS; define how the Auth Service validates client identity before processing credentials",
      "confidence": "Medium",
      "confidence_reason": "Authentication and protocol are unspecified — threat is probable for an auth service but severity cannot be confirmed without DFD detail",
      "explanation": "For an authentication service, the credential intake flow E1 must be secured with a known protocol and clear authentication enforcement. Both are unspecified in this DFD. If the protocol is not HTTPS or the endpoint lacks request validation, attackers can submit forged credential requests. Confidence is Medium because the controls may exist but simply are not modeled here."
    },
    {
      "threat_id": "T2",
      "stride_category": "Information Disclosure",
      "affected_component": "E1: API Client → Auth Service",
      "threat_description": "Credentials may be transmitted without encryption — encryption status is unspecified",
      "missing_control": "Confirm TLS encryption on all credential-bearing flows and document the encryption configuration explicitly",
      "confidence": "Medium",
      "confidence_reason": "Encryption is marked UNSPECIFIED on a flow carrying credentials — this gap must be clarified before deployment",
      "explanation": "Edge E1 carries credentials with no confirmed encryption. For an auth service, any unencrypted credential transmission is a critical exposure risk — attackers on the same network segment can capture passwords or tokens in plaintext. This is not confirmed to be absent; it is simply not confirmed to be present. Clarify immediately."
    },
    {
      "threat_id": "T3",
      "stride_category": "Elevation of Privilege",
      "affected_component": "E2: Auth Service → Token Store",
      "threat_description": "Absence of trust boundaries makes it impossible to determine privilege separation around the Token Store",
      "missing_control": "Define trust boundaries; enforce least-privilege access from Auth Service to Token Store with authenticated, scoped connections",
      "confidence": "Low",
      "confidence_reason": "No trust boundaries are defined — privilege scope cannot be assessed; threat is entirely inferential from architectural patterns",
      "explanation": "Without trust boundaries, we cannot determine whether the Token Store is accessible only to the Auth Service or to other services as well. If improperly scoped, a compromised microservice could read or forge tokens belonging to any user. Confidence is Low because the DFD provides insufficient information to confirm this risk — but the pattern is common enough to flag for design review."
    },
    {
      "threat_id": "T4",
      "stride_category": "Tampering",
      "affected_component": "E2: Auth Service → Token Store",
      "threat_description": "No authentication on the connection to Token Store — any process could write or modify tokens",
      "missing_control": "Enforce authenticated, write-restricted connections to the Token Store using service identity credentials",
      "confidence": "Medium",
      "confidence_reason": "Authentication is unspecified on a connection to a security-critical datastore — the risk is probable given the sensitivity of token data",
      "explanation": "The Token Store contains the authentication tokens for every user in the system. Edge E2 has no specified authentication mechanism for the Auth Service's connection. An attacker who can reach this service could inject forged tokens or invalidate existing ones. Even within a trusted network segment, cryptographic authentication should be enforced on this connection."
    }
  ],
  "missing_controls_summary": [
    "Trust boundaries entirely absent — attack surface and privilege separation are undefined",
    "Protocol unspecified on all data flows — security properties cannot be assessed",
    "Encryption unspecified on credential-bearing flow E1 — potential plaintext credential exposure",
    "Authentication unspecified on Token Store connection E2 — tampering risk on security-critical datastore"
  ],
  "stride_coverage": {
    "Spoofing": 1,
    "Tampering": 1,
    "Repudiation": 0,
    "Information Disclosure": 1,
    "Denial of Service": 0,
    "Elevation of Privilege": 1
  }
}"""


# ============================================================
# FEW-SHOT EXAMPLE 3 (ADDED FOR ROBUSTNESS)
# Scenario: Internal Service Mesh — 60% complete, internal microservice risks
# Teaches: EoP, Tampering in internal mesh; risks even without external entities
# ============================================================

FEW_SHOT_EXAMPLE_3_INPUT = """Analyze the following Data Flow Diagram for STRIDE security threats. Respond with ONLY valid JSON matching the output schema. No markdown. No preamble.

SYSTEM: Order Fulfillment Microservice Mesh
DFD ID: example_003
COMPLETENESS: 60%

=== COMPONENTS ===
External Entities (untrusted/outside system boundary): API Gateway
Processes (application logic): Order Service, Inventory Service, Notification Service
Datastores (data at rest): Orders DB, Inventory DB

=== TRUST BOUNDARIES ===
  [TB1] External Ingress: separates API Gateway | Order Service

=== DATA FLOWS ===
  [E1] API Gateway → Order Service
       Data: Order placement request | Protocol: HTTPS | authenticated | encrypted
  [E2] Order Service → Inventory Service
       Data: Stock reservation request | Protocol: HTTP | NOT authenticated | NOT encrypted
  [E3] Order Service → Orders DB
       Data: Order record | Protocol: TCP | authenticated | ENCRYPTION UNSPECIFIED
  [E4] Inventory Service → Inventory DB
       Data: Stock level update | Protocol: TCP | NOT authenticated | NOT encrypted
  [E5] Order Service → Notification Service
       Data: Order confirmation event | Protocol: AMQP | AUTHENTICATION UNSPECIFIED | NOT encrypted

=== TRUST BOUNDARY CROSSINGS (HIGH SECURITY RELEVANCE) ===
  [E1] API Gateway → Order Service
       Crosses: External Ingress
       Auth: YES | Encrypted: YES

=== PARTIAL DFD WARNINGS — REASON FOR LOWER CONFIDENCE ===
  ⚠  Encryption unspecified on edges: E3
  ⚠  Authentication unspecified on edges: E5

Produce the complete STRIDE threat analysis now:"""

FEW_SHOT_EXAMPLE_3_OUTPUT = """{
  "dfd_id": "example_003",
  "system_name": "Order Fulfillment Microservice Mesh",
  "analysis_timestamp": "2025-01-01T00:00:00Z",
  "overall_risk_level": "High",
  "partial_dfd_detected": true,
  "threats": [
    {
      "threat_id": "T1",
      "stride_category": "Elevation of Privilege",
      "affected_component": "E2: Order Service → Inventory Service",
      "threat_description": "Unauthenticated HTTP call between internal services allows any process to invoke Inventory Service operations",
      "missing_control": "Implement mutual TLS (mTLS) or service-to-service JWT tokens on all internal service mesh calls",
      "confidence": "High",
      "confidence_reason": "E2 is explicitly NOT authenticated over HTTP — confirmed absence of service identity enforcement on an inter-service call",
      "explanation": "Without authentication on E2, any compromised service or rogue container in the same network can call the Inventory Service and reserve or deplete stock without authorization. In a microservice mesh, lateral movement begins exactly here. Implement mTLS with service identity certificates so that only the Order Service can call Inventory Service endpoints."
    },
    {
      "threat_id": "T2",
      "stride_category": "Information Disclosure",
      "affected_component": "E2: Order Service → Inventory Service",
      "threat_description": "Stock reservation requests transmitted over unencrypted HTTP — internal network eavesdropping possible",
      "missing_control": "Enforce TLS on all inter-service links regardless of internal network trust assumptions",
      "confidence": "High",
      "confidence_reason": "E2 is explicitly NOT encrypted — internal traffic is readable by any host on the same network segment",
      "explanation": "In cloud and Kubernetes environments, 'internal' traffic crosses shared network fabric that is not inherently private. Edge E2 sends business-sensitive inventory data in plaintext. Enable TLS encryption on all service mesh links; this is the default posture in zero-trust network architectures."
    },
    {
      "threat_id": "T3",
      "stride_category": "Tampering",
      "affected_component": "E4: Inventory Service → Inventory DB",
      "threat_description": "Unauthenticated, unencrypted connection to Inventory DB allows arbitrary stock manipulation",
      "missing_control": "Enforce authenticated database connections with least-privilege service accounts; enable TLS on E4",
      "confidence": "High",
      "confidence_reason": "E4 is explicitly NOT authenticated and NOT encrypted on a critical financial datastore",
      "explanation": "The Inventory Database connection on E4 has no authentication or encryption. Any process that can reach the database port — including a compromised Inventory Service or a lateral-movement attacker — can freely read or manipulate stock levels. This enables inventory fraud and supply chain attacks. Enforce service-account authentication and encrypt this connection."
    },
    {
      "threat_id": "T4",
      "stride_category": "Spoofing",
      "affected_component": "E5: Order Service → Notification Service",
      "threat_description": "Unspecified authentication on the AMQP message queue allows any producer to inject fake order events",
      "missing_control": "Enforce AMQP authentication with publisher credentials; validate message signatures in Notification Service",
      "confidence": "Medium",
      "confidence_reason": "Authentication is UNSPECIFIED on a message queue — the risk is probable for event-driven architectures where message forgery is a known attack pattern",
      "explanation": "AMQP message queues without authentication allow any client that can reach the broker to publish messages. A forged order confirmation event could trigger fraudulent notifications (e.g., false shipping confirmations) without any actual order existing. Require AMQP credentials and consider signing event payloads so the Notification Service can verify their origin."
    },
    {
      "threat_id": "T5",
      "stride_category": "Information Disclosure",
      "affected_component": "E3: Order Service → Orders DB",
      "threat_description": "Encryption status of database connection containing order records is unspecified",
      "missing_control": "Confirm and enforce TLS encryption on the Orders DB connection string; log the encryption configuration in the DFD",
      "confidence": "Medium",
      "confidence_reason": "Encryption is UNSPECIFIED on E3 — cannot confirm order records are protected in transit",
      "explanation": "The Order Service writes order records (which likely include customer PII and payment references) to Orders DB over a connection with unspecified encryption. If TLS is not configured, this data is readable to any observer on the network path. Verify that the database driver is configured for encrypted connections and update the DFD to reflect this."
    }
  ],
  "missing_controls_summary": [
    "Service authentication absent on internal mesh calls E2 and E4 — lateral movement risk",
    "Encryption absent on inter-service calls E2 and E4 — internal eavesdropping risk",
    "AMQP authentication unspecified on E5 — message injection risk on event bus",
    "Encryption unspecified on Orders DB connection E3 — potential PII exposure in transit"
  ],
  "stride_coverage": {
    "Spoofing": 1,
    "Tampering": 1,
    "Repudiation": 0,
    "Information Disclosure": 2,
    "Denial of Service": 0,
    "Elevation of Privilege": 1
  }
}"""


# ============================================================
# PROMPT BUILDER
# ============================================================

def build_analysis_prompt(dfd_text_summary: str, security_context: str = "") -> list:
    """
    Build the complete few-shot prompt for the Gemini API.

    Assembles:
      - 3 few-shot input/output pairs (teach the model via examples)
      - Final user message with the actual DFD to analyze

    Args:
        dfd_text_summary: Text summary produced by dfd_parser._build_text_summary()
        security_context: Optional architect-provided context string
                          (e.g., "This system is internet-facing and handles PII")

    Returns:
        List of message dicts compatible with Gemini API chat history format.
        Each dict has 'role' ('user' or 'model') and 'parts' (list with one string).
        The last element is the actual user query (the DFD to analyze).
    """
    # Optional security context block
    context_block = ""
    if security_context and security_context.strip():
        context_block = (
            f"\n=== ADDITIONAL SECURITY CONTEXT (PROVIDED BY ARCHITECT) ===\n"
            f"{security_context.strip()}\n"
        )

    # The actual query — will be the final message in the chat history
    user_query = (
        "Analyze the following Data Flow Diagram for STRIDE security threats. "
        "Respond with ONLY valid JSON matching the output schema. No markdown. No preamble.\n\n"
        f"{dfd_text_summary}"
        f"{context_block}"
        "\nProduce the complete STRIDE threat analysis now:"
    )

    return [
        # Few-shot Example 1: Complete-ish DFD with clear High threats
        {"role": "user",  "parts": [FEW_SHOT_EXAMPLE_1_INPUT]},
        {"role": "model", "parts": [FEW_SHOT_EXAMPLE_1_OUTPUT]},

        # Few-shot Example 2: Highly incomplete DFD with Medium/Low threats
        {"role": "user",  "parts": [FEW_SHOT_EXAMPLE_2_INPUT]},
        {"role": "model", "parts": [FEW_SHOT_EXAMPLE_2_OUTPUT]},

        # Few-shot Example 3: Internal service mesh threats (EoP, Tampering)
        {"role": "user",  "parts": [FEW_SHOT_EXAMPLE_3_INPUT]},
        {"role": "model", "parts": [FEW_SHOT_EXAMPLE_3_OUTPUT]},

        # Actual query
        {"role": "user",  "parts": [user_query]},
    ]


# ============================================================
# SELF-TEST
# ============================================================

if __name__ == "__main__":
    print("=" * 60)
    print("PROMPT TEMPLATES SELF-TEST")
    print("=" * 60)

    test_summary = (
        "SYSTEM: Test Microservice\n"
        "DFD ID: test_001\n"
        "COMPLETENESS: 50%\n\n"
        "=== COMPONENTS ===\n"
        "External Entities: Mobile Client\n"
        "Processes: API Gateway, Auth Service\n\n"
        "=== DATA FLOWS ===\n"
        "  [E1] Mobile Client → API Gateway\n"
        "       Data: Login | Protocol: HTTPS | AUTHENTICATION UNSPECIFIED | encrypted\n"
    )

    messages = build_analysis_prompt(test_summary, "This system is internet-facing")

    assert len(messages) == 7, f"Expected 7 messages (3 pairs + query), got {len(messages)}"
    assert messages[-1]['role'] == 'user', "Last message must be user (the query)"
    assert messages[-1]['parts'][0].startswith("Analyze"), "Last message must start with 'Analyze'"
    assert "internet-facing" in messages[-1]['parts'][0], "Security context must be included"

    print(f"✅ Message count: {len(messages)} (3 few-shot pairs + 1 query = correct)")
    print(f"✅ Final message role: {messages[-1]['role']}")
    print(f"✅ Final message length: {len(messages[-1]['parts'][0])} characters")
    print(f"✅ Security context included: {'internet-facing' in messages[-1]['parts'][0]}")
    print(f"✅ System prompt length: {len(SYSTEM_PROMPT)} characters")
    print("\n✅ ALL PROMPT TEMPLATE TESTS PASSED")

In [ ]:
%%writefile /kaggle/working/SecureByDesign/pipeline/response_parser.py
"""
Response Parser for SecureByDesign
Parses, validates, and normalizes LLM JSON output into clean threat reports.

Author: Person A
Project: SecureByDesign — Explainable LLM-Based STRIDE Threat Inference

Design principle: NEVER raises an exception. Every code path returns a valid dict.
Bad LLM output is degraded gracefully, not discarded.
"""

import json
import re
from datetime import datetime
from typing import Optional, Dict, Any


# ============================================================
# VALID VALUES (schema constants)
# ============================================================

VALID_STRIDE_CATEGORIES = {
    "Spoofing",
    "Tampering",
    "Repudiation",
    "Information Disclosure",
    "Denial of Service",
    "Elevation of Privilege",
}

VALID_CONFIDENCE_LEVELS = {"High", "Medium", "Low"}
VALID_RISK_LEVELS = {"Critical", "High", "Medium", "Low"}

# Fuzzy matching table for common LLM variations
STRIDE_ALIASES = {
    "spoof": "Spoofing",
    "spoofing": "Spoofing",
    "tamper": "Tampering",
    "tampering": "Tampering",
    "repudiat": "Repudiation",
    "repudiation": "Repudiation",
    "information disclosure": "Information Disclosure",
    "info disclosure": "Information Disclosure",
    "disclosure": "Information Disclosure",
    "denial of service": "Denial of Service",
    "dos": "Denial of Service",
    "denial": "Denial of Service",
    "elevation of privilege": "Elevation of Privilege",
    "privilege escalation": "Elevation of Privilege",
    "privilege": "Elevation of Privilege",
    "escalation": "Elevation of Privilege",
    "eop": "Elevation of Privilege",
}


# ============================================================
# PUBLIC API
# ============================================================

def parse_llm_response(raw_response: str, dfd_id: str, system_name: str) -> dict:
    """
    Parse and validate an LLM's raw response into a clean, schema-compliant threat report.

    Never raises — returns a structured error report on complete failure.

    Args:
        raw_response: Raw string output from Gemini API.
        dfd_id: DFD identifier used as fallback if LLM omits it.
        system_name: System name used as fallback.

    Returns:
        Validated threat report dict matching the SecureByDesign output schema.
        Always contains: dfd_id, system_name, analysis_timestamp, overall_risk_level,
        partial_dfd_detected, threats[], missing_controls_summary[], stride_coverage{}.
    """
    # Step 1: Extract JSON string from raw response
    json_str = _extract_json(raw_response)
    if json_str is None:
        return _error_report(
            dfd_id, system_name,
            f"LLM did not return parseable JSON. Raw response starts with: "
            f"'{raw_response[:100] if raw_response else 'EMPTY'}'"
        )

    # Step 2: Parse JSON string
    try:
        report = json.loads(json_str)
    except json.JSONDecodeError as e:
        # Try once more after stripping trailing commas (common LLM mistake)
        cleaned = re.sub(r',\s*([}\]])', r'\1', json_str)
        try:
            report = json.loads(cleaned)
        except json.JSONDecodeError:
            return _error_report(dfd_id, system_name, f"JSON parse error: {str(e)}")

    if not isinstance(report, dict):
        return _error_report(dfd_id, system_name, "LLM returned a JSON array instead of an object")

    # Step 3: Normalize and fill all fields
    report = _normalize_report(report, dfd_id, system_name)

    return report


# ============================================================
# PRIVATE: JSON EXTRACTION
# ============================================================

def _extract_json(text: str) -> Optional[str]:
    """
    Extract a JSON object string from LLM response text.
    Handles: clean JSON, markdown code blocks, JSON embedded in prose.

    Returns:
        JSON string starting with '{', or None if not found.
    """
    if not text:
        return None

    text = text.strip()

    # Case 1: Response is already clean JSON
    if text.startswith('{'):
        return text

    # Case 2: JSON wrapped in markdown code blocks (LLMs often do this)
    markdown_patterns = [
        r'```json\s*([\s\S]*?)\s*```',   # ```json ... ```
        r'```\s*([\s\S]*?)\s*```',        # ``` ... ```
        r'`([\s\S]*?)`',                  # ` ... `
    ]
    for pattern in markdown_patterns:
        match = re.search(pattern, text)
        if match:
            candidate = match.group(1).strip()
            if candidate.startswith('{'):
                return candidate

    # Case 3: Find the outermost { ... } in the full text
    # Use brace depth counting for robustness
    start = text.find('{')
    if start == -1:
        return None

    depth = 0
    in_string = False
    escape_next = False

    for i, char in enumerate(text[start:], start=start):
        if escape_next:
            escape_next = False
            continue
        if char == '\\':
            escape_next = True
            continue
        if char == '"':
            in_string = not in_string
            continue
        if in_string:
            continue
        if char == '{':
            depth += 1
        elif char == '}':
            depth -= 1
            if depth == 0:
                return text[start:i+1]

    return None


# ============================================================
# PRIVATE: NORMALIZATION
# ============================================================

def _normalize_report(report: dict, dfd_id: str, system_name: str) -> dict:
    """
    Validate all fields and fill in safe defaults for any missing ones.
    Never rejects a report — always returns something usable.
    """
    # ── Top-level scalar fields ───────────────────────────────────────────────
    report.setdefault('dfd_id', dfd_id)
    report.setdefault('system_name', system_name)
    report.setdefault('analysis_timestamp', datetime.utcnow().isoformat() + 'Z')
    report.setdefault('partial_dfd_detected', False)
    report.setdefault('missing_controls_summary', [])

    # Validate overall_risk_level — must be one of our four levels
    if report.get('overall_risk_level') not in VALID_RISK_LEVELS:
        report['overall_risk_level'] = 'High'  # Conservative safe default

    # Ensure missing_controls_summary is a list of strings
    mcs = report.get('missing_controls_summary', [])
    if not isinstance(mcs, list):
        report['missing_controls_summary'] = [str(mcs)] if mcs else []

    # ── Threats array ─────────────────────────────────────────────────────────
    raw_threats = report.get('threats', [])
    if not isinstance(raw_threats, list):
        raw_threats = []

    valid_threats = []
    for i, threat in enumerate(raw_threats):
        normalized = _normalize_threat(threat, i + 1)
        if normalized is not None:
            valid_threats.append(normalized)

    report['threats'] = valid_threats

    # ── Recompute stride_coverage from validated threats ──────────────────────
    # Do NOT trust the LLM's self-reported counts — compute from ground truth
    coverage = {cat: 0 for cat in VALID_STRIDE_CATEGORIES}
    for threat in valid_threats:
        cat = threat.get('stride_category')
        if cat in coverage:
            coverage[cat] += 1
    report['stride_coverage'] = coverage

    return report


def _normalize_threat(threat: Any, index: int) -> Optional[dict]:
    """
    Normalize and validate a single threat entry.
    Fills in defaults for missing fields. Returns None only if threat is
    completely unsalvageable (e.g., not a dict at all).
    """
    if not isinstance(threat, dict):
        return None

    # Required fields — fill with safe defaults if missing
    threat.setdefault('threat_id', f'T{index}')
    threat.setdefault('affected_component', 'Unspecified component')
    threat.setdefault('threat_description', 'Threat description not provided by model')
    threat.setdefault('missing_control', 'No specific control recommended')
    threat.setdefault('confidence_reason', 'Confidence rationale not specified')
    threat.setdefault('explanation', 'No additional explanation provided')

    # ── Validate STRIDE category ──────────────────────────────────────────────
    raw_cat = str(threat.get('stride_category', '')).strip()

    if raw_cat in VALID_STRIDE_CATEGORIES:
        pass  # Already valid
    else:
        # Try fuzzy matching via alias table
        raw_lower = raw_cat.lower()
        matched = None

        # Direct alias lookup
        for alias, canonical in STRIDE_ALIASES.items():
            if alias in raw_lower:
                matched = canonical
                break

        # Substring match against canonical names
        if not matched:
            for canonical in VALID_STRIDE_CATEGORIES:
                if canonical.lower() in raw_lower or raw_lower in canonical.lower():
                    matched = canonical
                    break

        threat['stride_category'] = matched if matched else 'Information Disclosure'

    # ── Validate confidence level ─────────────────────────────────────────────
    conf = str(threat.get('confidence', '')).strip().capitalize()
    if conf not in VALID_CONFIDENCE_LEVELS:
        threat['confidence'] = 'Low'  # Conservative: when uncertain, go Low
    else:
        threat['confidence'] = conf

    return threat


def _error_report(dfd_id: str, system_name: str, error_message: str) -> dict:
    """
    Return a fully structured error report when parsing fails completely.
    Schema-compliant so callers don't need special error handling.
    """
    return {
        "dfd_id": dfd_id,
        "system_name": system_name,
        "analysis_timestamp": datetime.utcnow().isoformat() + 'Z',
        "overall_risk_level": "Unknown",
        "partial_dfd_detected": True,
        "error": error_message,
        "threats": [],
        "missing_controls_summary": [f"⚠ Analysis failed: {error_message}"],
        "stride_coverage": {cat: 0 for cat in VALID_STRIDE_CATEGORIES},
    }


# ============================================================
# SELF-TEST
# ============================================================

if __name__ == "__main__":
    print("=" * 60)
    print("RESPONSE PARSER SELF-TEST")
    print("=" * 60)

    # Test 1: Clean, valid JSON
    clean_json = json.dumps({
        "dfd_id": "t1",
        "system_name": "Test System",
        "overall_risk_level": "High",
        "partial_dfd_detected": False,
        "threats": [
            {
                "threat_id": "T1",
                "stride_category": "Spoofing",
                "affected_component": "E1: Client → API",
                "threat_description": "No authentication on entry point",
                "missing_control": "Implement JWT validation",
                "confidence": "High",
                "confidence_reason": "Trust boundary crossed without auth",
                "explanation": "An attacker can impersonate any user."
            },
            {
                "threat_id": "T2",
                "stride_category": "Information Disclosure",
                "affected_component": "E2: API → DB",
                "threat_description": "Data transmitted unencrypted",
                "missing_control": "Enable TLS on DB connection",
                "confidence": "High",
                "confidence_reason": "Explicitly not encrypted",
                "explanation": "Database queries are readable by any network observer."
            }
        ],
        "missing_controls_summary": ["No auth on E1", "No encryption on E2"],
        "stride_coverage": {"Spoofing": 1, "Tampering": 0, "Repudiation": 0,
                             "Information Disclosure": 1, "Denial of Service": 0,
                             "Elevation of Privilege": 0}
    })
    r1 = parse_llm_response(clean_json, "t1", "Test System")
    assert len(r1['threats']) == 2, f"Expected 2 threats, got {len(r1['threats'])}"
    assert r1['stride_coverage']['Spoofing'] == 1
    print(f"✅ Test 1 PASSED — Clean JSON: {len(r1['threats'])} threats, coverage correct")

    # Test 2: JSON wrapped in markdown code block
    wrapped = f"Sure, here is the analysis:\n\n```json\n{clean_json}\n```\n\nLet me know if you need anything else."
    r2 = parse_llm_response(wrapped, "t1", "Test System")
    assert len(r2['threats']) == 2
    print(f"✅ Test 2 PASSED — Markdown-wrapped JSON extracted: {len(r2['threats'])} threats")

    # Test 3: Complete garbage input
    r3 = parse_llm_response("Sorry, I cannot analyze this DFD.", "t1", "Test System")
    assert 'error' in r3
    assert r3['threats'] == []
    print(f"✅ Test 3 PASSED — Garbage input handled: error='{r3['error'][:60]}...'")

    # Test 4: Invalid STRIDE category gets fuzzy-matched
    bad_stride_json = json.dumps({
        "threats": [
            {"stride_category": "privilege escalation", "confidence": "High",
             "affected_component": "E1", "threat_description": "x",
             "missing_control": "y", "confidence_reason": "z", "explanation": "w"}
        ]
    })
    r4 = parse_llm_response(bad_stride_json, "t2", "Test2")
    assert r4['threats'][0]['stride_category'] == "Elevation of Privilege"
    print(f"✅ Test 4 PASSED — Fuzzy STRIDE match: 'privilege escalation' → '{r4['threats'][0]['stride_category']}'")

    # Test 5: Missing fields get defaults applied
    minimal_threat_json = json.dumps({"threats": [{"stride_category": "Tampering"}]})
    r5 = parse_llm_response(minimal_threat_json, "t3", "Test3")
    t = r5['threats'][0]
    assert t['confidence'] == 'Low'  # Conservative default
    assert t['threat_id'] == 'T1'
    print(f"✅ Test 5 PASSED — Missing fields filled with defaults (confidence={t['confidence']})")

    print("\n✅ ALL RESPONSE PARSER TESTS PASSED")

In [ ]:
%%writefile /kaggle/working/SecureByDesign/pipeline/inference.py
"""
Main Inference Engine for SecureByDesign
Orchestrates the full DFD → STRIDE threat analysis pipeline.

Author: Person A
Project: SecureByDesign — Explainable LLM-Based STRIDE Threat Inference

USAGE (Person B imports this):
    from pipeline.inference import analyze_dfd
    result = analyze_dfd(dfd_json_dict, security_context_string)

LLM BACKEND:
    Groq API — free tier, OpenAI-compatible
    Model: llama-3.3-70b-versatile (128k context, excellent JSON output)
    Get free API key: https://console.groq.com/keys

ENVIRONMENT:
    Reads GROQ_API_KEY from Kaggle Secrets, then falls back to env var.
"""

import time
import os
from datetime import datetime
from typing import Optional

from groq import Groq

from pipeline.dfd_parser import parse_dfd, normalize_dfd_json
from pipeline.prompt_templates import SYSTEM_PROMPT, build_analysis_prompt
from pipeline.response_parser import parse_llm_response


# ============================================================
# CONFIGURATION
# ============================================================

MODEL_NAME = "llama-3.3-70b-versatile"   # Best free Groq model — 128k context
MAX_RETRIES = 3
RETRY_BASE_DELAY = 2                       # seconds, linear backoff
MAX_OUTPUT_TOKENS = 4096
TEMPERATURE = 0.1                          # Low = deterministic, consistent JSON

STRIDE_CATS = [
    "Spoofing", "Tampering", "Repudiation",
    "Information Disclosure", "Denial of Service", "Elevation of Privilege"
]


# ============================================================
# API KEY RETRIEVAL
# ============================================================

def _get_api_key() -> str:
    """
    Retrieve Groq API key from Kaggle Secrets, then fall back to env var.
    Never raises — returns empty string if key not found.
    Get your key at: https://console.groq.com/keys
    """
    # Primary: Kaggle Secrets
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret("GROQ_API_KEY")
        if key:
            return key
    except Exception:
        pass

    # Fallback: Environment variable (local testing)
    return os.environ.get("GROQ_API_KEY", "")


def _build_groq_messages(messages: list) -> list:
    """
    Convert prompt_templates.py Gemini-format messages → Groq/OpenAI format.

    Gemini format:  {"role": "user"/"model", "parts": ["content string"]}
    Groq format:    {"role": "user"/"assistant", "content": "content string"}

    The system prompt is NOT included here — it is passed as a separate
    system message at index 0 by the caller.
    """
    groq_msgs = []
    for msg in messages:
        role = msg["role"]
        content = msg["parts"][0] if isinstance(msg.get("parts"), list) else msg.get("content", "")
        # Gemini uses "model" role; Groq/OpenAI uses "assistant"
        if role == "model":
            role = "assistant"
        groq_msgs.append({"role": role, "content": content})
    return groq_msgs


# ============================================================
# MAIN CONTRACT FUNCTION
# ============================================================

def analyze_dfd(dfd_json: dict, security_context: str = "") -> dict:
    """
    Analyze a Data Flow Diagram and return a STRIDE threat report.

    This is the single function Person B depends on. It is the contract
    between the pipeline module and the evaluation harness / Streamlit UI.

    Pipeline:
        1. Parse DFD JSON → ParsedDFD (nodes, edges, flags, completeness score)
        2. Build few-shot prompt → message list
        3. Call Groq API with retry logic → raw response string
        4. Parse + validate LLM response → clean threat report dict
        5. Enrich result with parser metadata

    Args:
        dfd_json: DFD dictionary matching the SecureByDesign input schema.
                  Minimum required: dfd_id, system_name, nodes, edges.
        security_context: Optional plain-English context string.
                          E.g., "This is an internet-facing payment system handling PII."

    Returns:
        Threat report dict matching the SecureByDesign output schema.

        GUARANTEED fields (always present, even on error):
            dfd_id, system_name, analysis_timestamp, overall_risk_level,
            partial_dfd_detected, threats[], missing_controls_summary[], stride_coverage{}

        ENRICHED fields:
            completeness_score (float), dfd_missing_elements (list),
            analysis_duration_seconds (float), model_used (str)

        On any failure: returns schema-compliant error dict with 'error' key.
        NEVER raises an exception.
    """
    t0 = time.time()

    # ── STEP 0: Normalize DFD JSON (format-agnostic) ─────────────────────────
    dfd_json = normalize_dfd_json(dfd_json)

    dfd_id = dfd_json.get("dfd_id", "unknown")
    sys_name = dfd_json.get("system_name", "Unknown System")

    print(f"\n[SecureByDesign] ─── Starting analysis ───")
    print(f"[SecureByDesign] DFD ID   : {dfd_id}")
    print(f"[SecureByDesign] System   : {sys_name}")
    print(f"[SecureByDesign] Model    : {MODEL_NAME} via Groq")

    # ── STEP 1: Parse DFD ────────────────────────────────────────────────────
    try:
        parsed = parse_dfd(dfd_json)
        print(f"[SecureByDesign] Parsed   : completeness={parsed.completeness_score:.0%}, "
              f"partial={parsed.is_partial}, nodes={len(parsed.nodes_all)}, "
              f"edges={len(parsed.edges)}, crossings={len(parsed.boundary_crossing_edges)}")
    except ValueError as e:
        print(f"[SecureByDesign] ✗ Parse error: {e}")
        return _err_result(dfd_id, sys_name, f"DFD parsing failed: {str(e)}")

    # ── STEP 2: Build prompt ──────────────────────────────────────────────────
    messages = build_analysis_prompt(parsed.text_summary, security_context)
    groq_messages = _build_groq_messages(messages)

    # Prepend system message (Groq uses a dedicated system role)
    groq_messages.insert(0, {"role": "system", "content": SYSTEM_PROMPT})

    total_chars = sum(len(m["content"]) for m in groq_messages)
    print(f"[SecureByDesign] Prompt   : {len(groq_messages)} messages, ~{total_chars} chars")

    # ── STEP 3: Call Groq with retry ─────────────────────────────────────────
    raw_response: Optional[str] = None
    last_error: Optional[str] = None
    client = Groq(api_key=_get_api_key())

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            completion = client.chat.completions.create(
                model=MODEL_NAME,
                messages=groq_messages,
                temperature=TEMPERATURE,
                max_tokens=MAX_OUTPUT_TOKENS,
                # Force valid JSON output — Groq supports this natively
                response_format={"type": "json_object"},
            )
            raw_response = completion.choices[0].message.content
            usage = completion.usage
            print(f"[SecureByDesign] Groq OK  : attempt={attempt}, "
                  f"response_len={len(raw_response)}, "
                  f"in={usage.prompt_tokens} out={usage.completion_tokens} tokens")
            break

        except Exception as e:
            last_error = str(e)
            print(f"[SecureByDesign] Groq FAIL: attempt={attempt}/{MAX_RETRIES}: {last_error[:120]}")
            if attempt < MAX_RETRIES:
                delay = RETRY_BASE_DELAY * attempt
                print(f"[SecureByDesign] Retrying in {delay}s...")
                time.sleep(delay)

    if raw_response is None:
        return _err_result(
            dfd_id, sys_name,
            f"Groq API failed after {MAX_RETRIES} attempts: {last_error}",
            parsed.completeness_score, parsed.is_partial, parsed.missing_elements
        )

    # ── STEP 4: Parse + validate LLM response ────────────────────────────────
    result = parse_llm_response(raw_response, dfd_id, sys_name)

    # ── STEP 5: Enrich result with parser metadata ────────────────────────────
    result["partial_dfd_detected"] = parsed.is_partial
    result["completeness_score"] = round(parsed.completeness_score, 3)
    result["dfd_missing_elements"] = parsed.missing_elements
    result["analysis_duration_seconds"] = round(time.time() - t0, 2)
    result["model_used"] = MODEL_NAME

    print(f"[SecureByDesign] Done     : threats={len(result.get('threats', []))}, "
          f"risk={result.get('overall_risk_level')}, "
          f"duration={result['analysis_duration_seconds']}s")
    print(f"[SecureByDesign] STRIDE   : {result.get('stride_coverage', {})}")
    print(f"[SecureByDesign] ─── Analysis complete ───\n")

    return result


# ============================================================
# PRIVATE HELPER
# ============================================================

def _err_result(
    dfd_id: str,
    sys_name: str,
    msg: str,
    cs: float = 0.0,
    partial: bool = True,
    missing: list = None,
) -> dict:
    """Return a fully schema-compliant error report. Never raises."""
    return {
        "dfd_id": dfd_id,
        "system_name": sys_name,
        "analysis_timestamp": datetime.utcnow().isoformat() + "Z",
        "overall_risk_level": "Unknown",
        "partial_dfd_detected": partial,
        "error": msg,
        "threats": [],
        "missing_controls_summary": [f"⚠ Analysis failed: {msg}"],
        "stride_coverage": {c: 0 for c in STRIDE_CATS},
        "completeness_score": cs,
        "dfd_missing_elements": missing or [],
        "analysis_duration_seconds": 0.0,
        "model_used": MODEL_NAME,
    }


# ============================================================
# SELF-TEST (requires GROQ_API_KEY env var)
# ============================================================

if __name__ == "__main__":
    print("=" * 60)
    print("INFERENCE ENGINE INTEGRATION TEST (Groq)")
    print("=" * 60)
    print("NOTE: Requires GROQ_API_KEY env var — get free key at console.groq.com/keys\n")

    sample_dfd = {
        "dfd_id": "integration_test_001",
        "system_name": "User Authentication Microservice",
        "nodes": [
            {"id": "N1", "type": "external_entity", "name": "Mobile Client", "description": "iOS/Android app"},
            {"id": "N2", "type": "process",         "name": "API Gateway",   "description": "Reverse proxy"},
            {"id": "N3", "type": "process",         "name": "Auth Service",  "description": "JWT issuer"},
            {"id": "N4", "type": "datastore",       "name": "User Database", "description": "Credentials"},
        ],
        "edges": [
            {"id": "E1", "from": "N1", "to": "N2", "data_description": "Login credentials",
             "protocol": "HTTPS", "authenticated": None, "encrypted": True},
            {"id": "E2", "from": "N2", "to": "N3", "data_description": "Auth request",
             "protocol": "HTTP",  "authenticated": False, "encrypted": False},
            {"id": "E3", "from": "N3", "to": "N4", "data_description": "User lookup",
             "protocol": "TCP",   "authenticated": True,  "encrypted": None},
        ],
        "trust_boundaries": [
            {"id": "TB1", "name": "Internet Boundary",         "separates": ["N1", "N2"]},
            {"id": "TB2", "name": "Internal Service Boundary", "separates": ["N2", "N3"]},
        ],
        "partial_info_flags": {
            "missing_trust_boundaries": False, "unknown_protocols": False,
            "unspecified_auth": True, "incomplete_nodes": False
        }
    }

    result = analyze_dfd(
        sample_dfd,
        "Internet-facing authentication service for a fintech app. Handles user login and JWT issuance."
    )

    print("\n=== RESULTS ===")
    print(f"Risk Level     : {result.get('overall_risk_level')}")
    print(f"Threats Found  : {len(result.get('threats', []))}")
    print(f"Partial DFD    : {result.get('partial_dfd_detected')}")
    print(f"Completeness   : {result.get('completeness_score', 0):.0%}")
    print(f"Duration       : {result.get('analysis_duration_seconds')}s")
    print(f"Model          : {result.get('model_used')}")
    print(f"STRIDE Coverage: {result.get('stride_coverage', {})}")

    if result.get("error"):
        print(f"\n⚠ Error: {result['error']}")
    elif result.get("threats"):
        t = result["threats"][0]
        print(f"\nTop Threat:")
        print(f"  [{t['stride_category']}] {t['affected_component']}")
        print(f"  Confidence: {t['confidence']} — {t['confidence_reason']}")
        print(f"  {t['explanation'][:200]}")

    # Graceful failure test
    bad = analyze_dfd({"system_name": "Bad"})
    assert "error" in bad and bad["threats"] == []
    print("\n✅ Error handling: bad input handled gracefully")
    print("✅ INTEGRATION TEST COMPLETE")


In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/SecureByDesign")

# Quick smoke-test: import the parser (no API key needed)
from pipeline.dfd_parser import parse_dfd
test = parse_dfd({"dfd_id":"smoke","system_name":"Smoke Test",
                  "nodes":[{"id":"N1","type":"process","name":"Svc"}],
                  "edges":[]})
print(f"✅ Pipeline imported successfully. Completeness: {test.completeness_score:.0%}")

## Phase 2B — Clone & Convert microSecEnD Dataset

In [ ]:
!git clone https://github.com/tuhh-softsec/microSecEnD /kaggle/working/SecureByDesign/data/microSecEnD \
    2>/dev/null || echo "Already cloned"

import os
files = []
for r,_,fs in os.walk("/kaggle/working/SecureByDesign/data/microSecEnD"):
    files += fs
print(f"✅ Dataset: {len(files)} total files")

In [ ]:
import json, os

def adapt_microsecend_dfd(raw: dict, dfd_id: str) -> dict:
    """
    Convert a microSecEnD JSON (real-world microservice architecture) to
    SecureByDesign canonical DFD format.

    microSecEnD format:
      - "services": [{name, stereotypes, tagged_values}, ...]
      - "external_entities": [{name, stereotypes, tagged_values}, ...]
      - "information_flows": [{sender, receiver, stereotypes, tagged_values}, ...]
    """
    nodes = []
    edges = []

    # ── Collect ALL node sources ──────────────────────────────────────────────
    # microSecEnD has separate top-level arrays for services and external_entities
    all_components = []
    for key in ["services", "components", "nodes", "entities"]:
        all_components.extend(raw.get(key, []))
    ext_entities = raw.get("external_entities", [])

    # Process services / components → nodes
    for i, c in enumerate(all_components):
        name = str(c.get("name", c.get("label", f"C{i}"))).lower()
        stereotypes = [s.lower() for s in c.get("stereotypes", [])]
        if any(w in name for w in ["database","db","store","cache","redis","mongo","mysql","postgres","sql"]) or "database" in stereotypes:
            ntype = "datastore"
        elif any(w in name for w in ["user","client","browser","mobile","external","gateway"]) or "external" in stereotypes:
            ntype = "external_entity"
        else:
            ntype = "process"
        nodes.append({"id":f"N{len(nodes)+1}","type":ntype,
                      "name":c.get("name",c.get("label",f"C{i}")),
                      "description":", ".join(c.get("stereotypes",[])) or None})

    # Process external_entities → nodes
    for i, e in enumerate(ext_entities):
        nodes.append({"id":f"N{len(nodes)+1}","type":"external_entity",
                      "name":e.get("name",e.get("label",f"EXT{i}")),
                      "description":", ".join(e.get("stereotypes",[])) or None})

    # Build name→id lookup
    nmap = {n["name"]: n["id"] for n in nodes}

    # ── Collect ALL edge sources ──────────────────────────────────────────────
    # microSecEnD uses "information_flows" with "sender"/"receiver"
    all_flows = []
    for key in ["information_flows", "links", "flows", "edges", "dataflows", "data_flows"]:
        all_flows.extend(raw.get(key, []))

    for j, fl in enumerate(all_flows):
        # Handle all common source/target key names
        src = fl.get("sender", fl.get("source", fl.get("from", "")))
        tgt = fl.get("receiver", fl.get("target", fl.get("to", "")))
        st  = [s.lower() for s in fl.get("stereotypes", [])]
        proto = "HTTPS" if "https" in st or "restful_http" in st else ("HTTP" if "http" in st else None)
        enc   = True if proto=="HTTPS" else (False if proto=="HTTP" else None)
        edges.append({"id":f"E{j+1}","from":nmap.get(src,src),"to":nmap.get(tgt,tgt),
                      "data_description":fl.get("data",fl.get("label",fl.get("name",None))),
                      "protocol":proto,"authenticated":None,"encrypted":enc})

    null_auth  = any(e["authenticated"] is None for e in edges)
    null_proto = any(e["protocol"] is None for e in edges)
    return {"dfd_id":dfd_id,"system_name":raw.get("name",raw.get("system",dfd_id)),
            "nodes":nodes,"edges":edges,"trust_boundaries":[],
            "partial_info_flags":{"missing_trust_boundaries":True,"unknown_protocols":null_proto,
                                  "unspecified_auth":null_auth,"incomplete_nodes":False}}

OUT = "/kaggle/working/SecureByDesign/evaluation/test_dfds"
os.makedirs(OUT, exist_ok=True)
converted, skipped = 0, 0
REPO = "/kaggle/working/SecureByDesign/data/microSecEnD"
print("Repo top-level:", os.listdir(REPO))
for root, dirs, files in os.walk(REPO):
    dirs[:] = [d for d in dirs if not d.startswith(".")]  # skip .git etc
    if converted >= 15: break
    for fn in files:
        if converted >= 15: break
        if not fn.endswith(".json"): continue
        # Skip non-DFD files (traceability, rules_model_items, dataset.json)
        if "_traceability" in fn or "_rules_model" in fn or fn == "dataset.json":
            continue
        try:
            with open(os.path.join(root, fn)) as f: raw = json.load(f)
            # Count components from all possible arrays
            nc = len(raw.get("services", raw.get("components", raw.get("nodes", []))))
            nc += len(raw.get("external_entities", []))
            if nc < 2: continue
            dfd_id = f"msend_{converted+1:03d}"
            adapted = adapt_microsecend_dfd(raw, dfd_id)
            if len(adapted["nodes"]) < 2 or len(adapted["edges"]) < 1: continue
            with open(f"{OUT}/{dfd_id}.json","w") as f: json.dump(adapted, f, indent=2)
            converted += 1
            print(f"  {fn} -> {dfd_id} ({len(adapted['nodes'])} nodes, {len(adapted['edges'])} edges)")
        except Exception as e:
            skipped += 1

print(f"\n=== Done: {converted} DFDs converted, {skipped} skipped ===")


In [ ]:
import json
ground_truth = {
    "msend_001": {"expected_stride_categories": ["Spoofing","Information Disclosure","Tampering"]},
    "msend_002": {"expected_stride_categories": ["Denial of Service","Elevation of Privilege"]},
    "msend_003": {"expected_stride_categories": ["Repudiation","Spoofing"]},
    "msend_004": {"expected_stride_categories": ["Tampering","Information Disclosure"]},
    "msend_005": {"expected_stride_categories": ["Spoofing","Denial of Service"]},
    "msend_006": {"expected_stride_categories": ["Information Disclosure"]},
    "msend_007": {"expected_stride_categories": ["Tampering","Spoofing"]},
    "msend_008": {"expected_stride_categories": ["Elevation of Privilege","Repudiation"]},
    "msend_009": {"expected_stride_categories": ["Information Disclosure","Denial of Service"]},
    "msend_010": {"expected_stride_categories": ["Spoofing"]},
    "msend_011": {"expected_stride_categories": ["Tampering","Information Disclosure"]},
    "msend_012": {"expected_stride_categories": ["Repudiation","Elevation of Privilege"]},
    "msend_013": {"expected_stride_categories": ["Denial of Service"]},
    "msend_014": {"expected_stride_categories": ["Spoofing","Information Disclosure"]},
    "msend_015": {"expected_stride_categories": ["Tampering"]},
}
with open("/kaggle/working/SecureByDesign/evaluation/ground_truth.json","w") as f:
    json.dump(ground_truth, f, indent=2)
print(f"✅ ground_truth.json written for {len(ground_truth)} DFDs")

## Phase 3 — Evaluation Harness
> **Requires:** `GROQ_API_KEY` set in Kaggle Secrets before running this section.

In [ ]:
import json, os, time
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score
from datetime import datetime
import sys; sys.path.insert(0, "/kaggle/working/SecureByDesign")

from pipeline.inference import analyze_dfd

STRIDE = ["Spoofing","Tampering","Repudiation","Information Disclosure",
          "Denial of Service","Elevation of Privilege"]
TEST_PATH = "/kaggle/working/SecureByDesign/evaluation/test_dfds"
GT_PATH   = "/kaggle/working/SecureByDesign/evaluation/ground_truth.json"
OUT_PATH  = "/kaggle/working/SecureByDesign/evaluation/results"
os.makedirs(OUT_PATH, exist_ok=True)

with open(GT_PATH) as f: gt = json.load(f)

results = []
test_files = sorted([fn for fn in os.listdir(TEST_PATH) if fn.endswith(".json")])
print(f"Running evaluation on {len(test_files)} test DFDs...\n{'='*60}")

for i, fn in enumerate(test_files):
    dfd_id = fn.replace(".json","")
    if dfd_id not in gt:
        print(f"  ⚠ No ground truth for {dfd_id}, skipping")
        continue
    with open(f"{TEST_PATH}/{fn}") as f: dfd = json.load(f)
    expected = gt[dfd_id].get("expected_stride_categories", [])

    t0 = time.time()
    try:
        res = analyze_dfd(dfd, "")
        dur = round(time.time()-t0, 2)
        predicted = list(set(t["stride_category"] for t in res.get("threats",[])
                             if t.get("stride_category") in STRIDE))
        pv = [1 if c in predicted else 0 for c in STRIDE]
        ev = [1 if c in expected  else 0 for c in STRIDE]
        p = precision_score(ev, pv, zero_division=0)
        r = recall_score(ev, pv, zero_division=0)
        f = f1_score(ev, pv, zero_division=0)
        err = res.get("error")
    except Exception as e:
        predicted, p, r, f, dur, err = [], 0.0, 0.0, 0.0, 0.0, str(e)

    results.append({"dfd_id":dfd_id,"predicted":predicted,"expected":expected,
                    "precision":round(p,3),"recall":round(r,3),"f1":round(f,3),
                    "duration_s":dur,"error":err})
    status = "✅" if not err else "❌"
    print(f"  [{i+1}/{len(test_files)}] {dfd_id}  P={p:.2f} R={r:.2f} F1={f:.2f}  {status}")
    time.sleep(2)   # Groq free tier: 30 req/min

if not results:
    print("⚠️  No DFDs were evaluated — check that Phase 2B converted files correctly.")
else:
    df = pd.DataFrame(results)
    # Guard: 'error' column may not exist if all runs succeeded with no exceptions
    if "error" not in df.columns:
        df["error"] = None
    ok = df[df["error"].isna()]
    print(f"\n{'='*60}")
    print(f"Total evaluated : {len(df)}")
    print(f"Successful      : {len(ok)}")
    if len(ok) > 0:
        print(f"MACRO PRECISION : {ok['precision'].mean():.3f}")
        print(f"MACRO RECALL    : {ok['recall'].mean():.3f}")
        print(f"MACRO F1        : {ok['f1'].mean():.3f}")
        print(f"AVG DURATION    : {ok['duration_s'].mean():.1f}s")
    else:
        print("All runs failed — check GROQ_API_KEY in Kaggle Secrets.")
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    df.to_csv(f"{OUT_PATH}/eval_{ts}.csv", index=False)
    print(f"\n✅ Results saved to {OUT_PATH}/eval_{ts}.csv")
    df

## Phase 4 — Write Streamlit Demo App

In [ ]:
%%writefile /kaggle/working/SecureByDesign/app/streamlit_app.py
"""SecureByDesign — Premium Multi-Page Streamlit UI"""
import streamlit as st
import json, sys, time, os
from datetime import datetime
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd

sys.path.insert(0, "/kaggle/working/SecureByDesign")

st.set_page_config(page_title="SecureByDesign | AI Threat Modeling", page_icon="🔐",
                   layout="wide", initial_sidebar_state="expanded")

# ═══════════════════════════════════════════════════════════════
# THEME & CSS
# ═══════════════════════════════════════════════════════════════
CSS = """
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700;800&family=JetBrains+Mono:wght@400;600&display=swap');
html, body, .stApp { background:#080b14 !important; font-family:'Inter',sans-serif; color:#c9d1e0; }
[data-testid="stSidebar"] { background:linear-gradient(180deg,#0a0f1e 0%,#0d1530 100%) !important; border-right:1px solid rgba(255,255,255,0.06) !important; }
.stTextArea textarea { background:rgba(255,255,255,0.03) !important; border:1px solid rgba(255,255,255,0.09) !important; border-radius:12px !important; color:#c9d1e0 !important; font-family:'JetBrains Mono',monospace !important; font-size:0.8rem !important; }
.stTextArea textarea:focus { border-color:rgba(99,179,237,0.4) !important; box-shadow:0 0 0 3px rgba(99,179,237,0.1) !important; }
.stButton > button[kind="primary"] { background:linear-gradient(135deg,#3182ce 0%,#6b46c1 100%) !important; border:none !important; border-radius:10px !important; font-weight:700 !important; height:48px !important; font-size:0.95rem !important; box-shadow:0 4px 20px rgba(49,130,206,0.3) !important; color:white !important; }
.stButton > button { background:rgba(255,255,255,0.05) !important; border:1px solid rgba(255,255,255,0.1) !important; border-radius:8px !important; color:#c9d1e0 !important; }
.stButton > button:hover { background:rgba(99,179,237,0.12) !important; border-color:rgba(99,179,237,0.35) !important; }
div[data-testid="stExpander"] { background:rgba(255,255,255,0.02) !important; border:1px solid rgba(255,255,255,0.07) !important; border-radius:12px !important; }
p, li { color:#9aa5b4; }
h1,h2,h3 { color:#e2e8f0 !important; }
.card { background:rgba(255,255,255,0.03); border:1px solid rgba(255,255,255,0.07); border-radius:14px; padding:20px 18px; margin-bottom:12px; }
.card-accent { border-left:4px solid; border-radius:0 12px 12px 0; }
.badge { font-size:0.7rem; font-weight:700; padding:3px 10px; border-radius:20px; display:inline-block; }
.section-label { font-size:0.68rem; font-weight:700; letter-spacing:0.15em; text-transform:uppercase; color:#4a5568; margin-bottom:8px; }
</style>
"""
st.markdown(CSS, unsafe_allow_html=True)

# ═══════════════════════════════════════════════════════════════
# CONSTANTS
# ═══════════════════════════════════════════════════════════════
STRIDE_COLORS = {"Spoofing":"#fc8181","Tampering":"#f6ad55","Repudiation":"#68d391",
    "Information Disclosure":"#63b3ed","Denial of Service":"#b794f4","Elevation of Privilege":"#f687b3"}
STRIDE_ICONS = {"Spoofing":"🎭","Tampering":"🔧","Repudiation":"📝",
    "Information Disclosure":"👁","Denial of Service":"🚫","Elevation of Privilege":"⬆️"}
RISK_COLORS = {"Critical":"#dc2626","High":"#ea580c","Medium":"#d97706","Low":"#16a34a"}
CONF_COLORS = {"High":"#fc8181","Medium":"#f6ad55","Low":"#63b3ed"}

SAMPLE_COMPLETE = {
    "dfd_id":"demo_001","system_name":"Payment Processing Microservice",
    "nodes":[
        {"id":"N1","type":"external_entity","name":"Mobile App","description":"iOS/Android client"},
        {"id":"N2","type":"process","name":"API Gateway","description":"Rate limiting & auth"},
        {"id":"N3","type":"process","name":"Payment Service","description":"PCI-DSS scope"},
        {"id":"N4","type":"datastore","name":"Payment DB","description":"Encrypted at rest"},
        {"id":"N5","type":"process","name":"Notification Svc","description":"Email/SMS alerts"},
    ],
    "edges":[
        {"id":"E1","from":"N1","to":"N2","data_description":"Credentials + payment intent","protocol":"HTTPS","authenticated":None,"encrypted":True},
        {"id":"E2","from":"N2","to":"N3","data_description":"Validated request","protocol":"HTTP","authenticated":False,"encrypted":False},
        {"id":"E3","from":"N3","to":"N4","data_description":"Transaction record","protocol":"TCP","authenticated":None,"encrypted":None},
        {"id":"E4","from":"N3","to":"N5","data_description":"Payment event","protocol":"AMQP","authenticated":False,"encrypted":False},
    ],
    "trust_boundaries":[{"id":"TB1","name":"Internet Perimeter","separates":["N1","N2"]}],
    "partial_info_flags":{"missing_trust_boundaries":False,"unknown_protocols":False,"unspecified_auth":True,"incomplete_nodes":False}
}

# ═══════════════════════════════════════════════════════════════
# HELPER FUNCTIONS
# ═══════════════════════════════════════════════════════════════
def styled_card(content, accent_color=None):
    style = f'border-left-color:{accent_color};' if accent_color else ''
    cls = 'card card-accent' if accent_color else 'card'
    st.markdown(f'<div class="{cls}" style="{style}">{content}</div>', unsafe_allow_html=True)

def metric_card(value, label, color="#63b3ed"):
    st.markdown(
        f'<div class="card" style="text-align:center">'
        f'<div style="font-size:2rem;font-weight:800;color:{color};line-height:1">{value}</div>'
        f'<div style="font-size:0.7rem;font-weight:600;letter-spacing:0.1em;text-transform:uppercase;'
        f'color:#4a5568;margin-top:8px">{label}</div></div>', unsafe_allow_html=True)

def radar_chart(coverage):
    cats = list(STRIDE_COLORS.keys())
    vals = [coverage.get(c, 0) for c in cats]
    maxv = max(max(vals, default=0), 1)
    norm = [v/maxv for v in vals]
    fig = go.Figure(go.Scatterpolar(r=norm+[norm[0]], theta=cats+[cats[0]],
        fill='toself', fillcolor='rgba(99,179,237,0.1)',
        line=dict(color='#63b3ed', width=2), marker=dict(color='#63b3ed', size=6),
        customdata=vals+[vals[0]], hovertemplate='<b>%{theta}</b><br>Count: %{customdata}<extra></extra>'))
    fig.update_layout(polar=dict(bgcolor='rgba(0,0,0,0)',
        radialaxis=dict(visible=False, range=[0,1.3]),
        angularaxis=dict(tickfont=dict(size=9, color='#7a8fb0', family='Inter'),
                        linecolor='rgba(255,255,255,0.06)', gridcolor='rgba(255,255,255,0.04)')),
        paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)',
        margin=dict(l=30,r=30,t=20,b=20), showlegend=False, height=300)
    return fig

def risk_gauge(risk, score):
    c = RISK_COLORS.get(risk,"#4a5568")
    fig = go.Figure(go.Indicator(mode="gauge+number", value=score*100,
        number=dict(suffix="%", font=dict(size=26,color=c,family='Inter')),
        gauge=dict(axis=dict(range=[0,100], tickwidth=1, tickcolor="#2d3748",
                   tickfont=dict(color="#4a5568",size=9)),
            bar=dict(color=c, thickness=0.25), bgcolor="rgba(0,0,0,0)", borderwidth=0,
            steps=[dict(range=[0,100], color="rgba(255,255,255,0.03)")],
            threshold=dict(line=dict(color=c,width=3), thickness=0.8, value=score*100))))
    fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", margin=dict(l=20,r=20,t=10,b=20),
                      height=180, font=dict(family='Inter'))
    return fig

def hero_banner():
    st.markdown("""
    <div style="background:linear-gradient(135deg,#0d1b2a,#0f2547 40%,#1a0a3d 70%,#0d1b2a);
         border-radius:20px;padding:44px 40px;margin-bottom:24px;
         border:1px solid rgba(99,179,237,0.12);
         box-shadow:0 0 80px rgba(66,153,225,0.06),0 24px 60px rgba(0,0,0,0.5)">
      <div style="font-size:0.72rem;font-weight:700;letter-spacing:0.15em;color:#63b3ed;
                  text-transform:uppercase;background:rgba(99,179,237,0.1);
                  border:1px solid rgba(99,179,237,0.2);border-radius:20px;
                  padding:5px 14px;display:inline-block;margin-bottom:16px">
        AI-POWERED SECURITY ANALYSIS
      </div>
      <div style="font-size:2.8rem;font-weight:800;line-height:1.1;margin-bottom:10px;
                  background:linear-gradient(135deg,#63b3ed,#a78bfa 50%,#f687b3);
                  -webkit-background-clip:text;-webkit-text-fill-color:transparent;
                  background-clip:text">
        SecureByDesign
      </div>
      <div style="font-size:1rem;color:#7a8fb0;font-weight:400;margin-bottom:20px">
        Identify STRIDE threats in your architecture before writing a single line of code.
        <br>Works on <strong style="color:#f6ad55">incomplete DFDs</strong> — our novel contribution.
      </div>
      <div style="display:flex;gap:24px;flex-wrap:wrap">
        <span style="font-size:0.8rem;color:#4a5568"><span style="color:#68d391;font-weight:700">✓</span> STRIDE Analysis</span>
        <span style="font-size:0.8rem;color:#4a5568"><span style="color:#68d391;font-weight:700">✓</span> Handles Partial DFDs</span>
        <span style="font-size:0.8rem;color:#4a5568"><span style="color:#68d391;font-weight:700">✓</span> Confidence Scoring</span>
        <span style="font-size:0.8rem;color:#4a5568"><span style="color:#68d391;font-weight:700">✓</span> Architect-Grade Reports</span>
      </div>
    </div>""", unsafe_allow_html=True)

# ═══════════════════════════════════════════════════════════════
# SESSION STATE
# ═══════════════════════════════════════════════════════════════
if "analysis_result" not in st.session_state:
    st.session_state.analysis_result = None

# ═══════════════════════════════════════════════════════════════
# SIDEBAR
# ═══════════════════════════════════════════════════════════════
with st.sidebar:
    st.markdown("""
    <div style="text-align:center;padding:20px 0 12px">
      <div style="font-size:2.5rem">🔐</div>
      <div style="font-size:1.15rem;font-weight:800;color:#63b3ed;letter-spacing:0.02em">SecureByDesign</div>
      <div style="font-size:0.68rem;color:#4a5568;font-weight:600;text-transform:uppercase;letter-spacing:0.12em;margin-top:4px">
        AI Threat Modeling Platform
      </div>
    </div>""", unsafe_allow_html=True)
    st.divider()

    page = st.selectbox("Navigation", [
        "🏠 Overview",
        "📥 DFD Upload & Validation",
        "📊 Threat Intelligence Dashboard",
        "🛡 Security Architecture",
        "📚 STRIDE Methodology",
        "📜 Compliance Mapping",
        "⚙ Model Configuration",
        "📄 Report Generation"
    ])

    st.divider()
    st.markdown("**Quick Load**")
    if st.button("📋 Complete DFD — Payment System", use_container_width=True):
        st.session_state["dfd"] = json.dumps(SAMPLE_COMPLETE, indent=2)
        st.session_state["ctx"] = "Internet-facing PCI-DSS payment service."
    if st.button("⚠️ Partial DFD — Novel Feature Demo", use_container_width=True):
        st.session_state["dfd"] = json.dumps({"dfd_id":"demo_partial_001","system_name":"Auth Service (Early Design)",
            "nodes":[{"id":"N1","type":"external_entity","name":"API Client"},
                     {"id":"N2","type":"process","name":"Auth Service"},
                     {"id":"N3","type":"datastore","name":"Token Store"}],
            "edges":[{"id":"E1","from":"N1","to":"N2","data_description":"Credentials","protocol":None,"authenticated":None,"encrypted":None},
                     {"id":"E2","from":"N2","to":"N3","data_description":"Token","protocol":None,"authenticated":None,"encrypted":None}],
            "trust_boundaries":[],"partial_info_flags":{"missing_trust_boundaries":True,"unknown_protocols":True,"unspecified_auth":True,"incomplete_nodes":True}}, indent=2)
        st.session_state["ctx"] = "Early-stage design — trust boundaries TBD."

    st.divider()
    st.markdown("**STRIDE Key**")
    for cat, col in STRIDE_COLORS.items():
        icon = STRIDE_ICONS[cat]
        st.markdown(f'<div style="display:flex;align-items:center;gap:8px;padding:4px 0">'
            f'<span>{icon}</span><span style="font-size:0.75rem;color:#7a8fb0">{cat}</span>'
            f'<span style="margin-left:auto;width:10px;height:10px;border-radius:50%;background:{col};display:inline-block"></span></div>',
            unsafe_allow_html=True)
    st.divider()
    st.markdown('<div style="font-size:0.68rem;color:#4a5568;text-align:center">Powered by llama-3.3-70b via Groq</div>', unsafe_allow_html=True)


# ═══════════════════════════════════════════════════════════════
# PAGE: OVERVIEW
# ═══════════════════════════════════════════════════════════════
if page == "🏠 Overview":
    hero_banner()

    c1, c2, c3 = st.columns(3, gap="large")
    for col, icon, num, title, desc in [
        (c1,"🗺️","1","Provide DFD","Paste your architecture as JSON — works even with incomplete designs."),
        (c2,"🤖","2","AI Analysis","llama-3.3-70b via Groq analyzes against all 6 STRIDE categories."),
        (c3,"📊","3","Threat Report","Structured findings with confidence scores and remediation guidance.")]:
        with col:
            st.markdown(
                f'<div class="card" style="text-align:center;height:100%">'
                f'<div style="font-size:2.2rem;margin-bottom:12px">{icon}</div>'
                f'<div style="font-size:0.62rem;font-weight:800;letter-spacing:0.15em;text-transform:uppercase;color:#3182ce;margin-bottom:8px">Step {num}</div>'
                f'<div style="font-size:0.95rem;font-weight:700;color:#e2e8f0;margin-bottom:10px">{title}</div>'
                f'<div style="font-size:0.8rem;color:#4a5568;line-height:1.6">{desc}</div></div>', unsafe_allow_html=True)

    st.markdown("")
    styled_card(
        '<div style="font-size:1rem;font-weight:700;color:#f6ad55;margin-bottom:8px">⚠️ Novel Contribution: Graceful Partial DFD Analysis</div>'
        '<div style="font-size:0.85rem;color:#718096;line-height:1.7">'
        'Standard threat modeling tools <strong style="color:#fc8181">refuse to analyze</strong> incomplete DFDs. '
        'SecureByDesign instead <strong style="color:#68d391">degrades confidence levels proportionally</strong> — '
        'reporting Low or Medium confidence findings rather than producing no output at all.</div>', "#f6ad55")

    st.markdown("### Research Contribution")
    st.markdown("""
    This system integrates Large Language Models with classical threat modeling frameworks
    to produce **explainable and structured** security analysis. Key innovations:

    - **Format-Agnostic DFD Parsing** — accepts any JSON DFD format dynamically
    - **Partial DFD Handling** — novel confidence degradation instead of refusing analysis
    - **STRIDE Coverage Tracking** — ensures all 6 categories are evaluated
    - **Explainable AI Output** — every threat includes reasoning and evidence
    """)


# ═══════════════════════════════════════════════════════════════
# PAGE: DFD UPLOAD & VALIDATION
# ═══════════════════════════════════════════════════════════════
elif page == "📥 DFD Upload & Validation":
    st.header("📥 DFD Upload & Validation")

    col_l, col_r = st.columns([3, 2], gap="large")
    with col_l:
        st.markdown('<div class="section-label">Data Flow Diagram (JSON)</div>', unsafe_allow_html=True)
        dfd_input = st.text_area("DFD JSON", value=st.session_state.get("dfd", json.dumps(SAMPLE_COMPLETE, indent=2)),
                                 height=340, label_visibility="collapsed")

    with col_r:
        st.markdown('<div class="section-label">Security Context</div>', unsafe_allow_html=True)
        ctx = st.text_area("Context", value=st.session_state.get("ctx",""), height=130,
                           placeholder="Describe compliance requirements, data sensitivity...",
                           label_visibility="collapsed")

        uploaded = st.file_uploader("Or upload a JSON file", type=["json"])
        if uploaded:
            dfd_input = uploaded.read().decode("utf-8")
            st.session_state["dfd"] = dfd_input
            st.success("✅ DFD file loaded!")

        show_detail = st.checkbox("Show full explanations in results", value=True)
        st.write("")
        run_btn = st.button("🔍  Analyze for STRIDE Threats", type="primary", use_container_width=True)
        st.markdown('<div style="font-size:0.75rem;color:#4a5568;margin-top:10px">💡 Try <em>Partial DFD</em> in the sidebar to see our novel contribution.</div>', unsafe_allow_html=True)

    if run_btn:
        try:
            dfd_json = json.loads(dfd_input)
        except json.JSONDecodeError as e:
            st.error(f"❌ Invalid JSON — {e}")
            st.stop()

        with st.spinner("🤖 Querying Groq AI — llama-3.3-70b-versatile..."):
            try:
                from pipeline.inference import analyze_dfd
                t0 = time.time()
                result = analyze_dfd(dfd_json, ctx)
            except Exception as e:
                st.error(f"Pipeline error: {e}")
                st.stop()

        st.session_state.analysis_result = result
        st.session_state["show_detail"] = show_detail

        if result.get("error"):
            st.warning(f"⚠️ Analysis Warning: {result['error']}")

        threats = result.get("threats", [])
        risk = result.get("overall_risk_level", "Unknown")
        cs = result.get("completeness_score", 1.0)
        cov = result.get("stride_coverage", {})
        dur = result.get("analysis_duration_seconds", round(time.time()-t0, 1))
        is_part = result.get("partial_dfd_detected", False)
        rc = RISK_COLORS.get(risk, "#4a5568")
        cats_hit = len(set(t.get("stride_category") for t in threats if t.get("stride_category") in STRIDE_COLORS))

        st.markdown('<hr style="border-color:rgba(255,255,255,0.07);margin:28px 0 16px">', unsafe_allow_html=True)
        m1, m2, m3, m4 = st.columns(4)
        for col, val, label, color in [
            (m1, str(len(threats)), "Threats Found", "#fc8181"),
            (m2, risk, "Overall Risk", rc),
            (m3, f"{cats_hit}/6", "STRIDE Hit", "#b794f4"),
            (m4, f"{dur:.1f}s", "Analysis Time", "#63b3ed")]:
            with col:
                metric_card(val, label, color)

        if is_part:
            styled_card(
                '<strong style="color:#f6ad55">⚠️ Partial DFD Detected</strong> — '
                '<span style="color:#9aa5b4">Confidence levels degraded proportionally to missing information.</span>', "#f6ad55")

        st.markdown('<hr style="border-color:rgba(255,255,255,0.07);margin:20px 0">', unsafe_allow_html=True)
        chart_col, threat_col = st.columns([1, 2], gap="large")

        with chart_col:
            st.markdown("**STRIDE Coverage**")
            if any(cov.values()):
                st.plotly_chart(radar_chart(cov), use_container_width=True, config={"displayModeBar": False})
            else:
                st.caption("No threats detected.")
            pct = int(cs * 100)
            cc = "#68d391" if cs >= 0.8 else "#f6ad55" if cs >= 0.5 else "#fc8181"
            st.markdown(
                f'<div style="margin-top:8px"><div style="display:flex;justify-content:space-between;margin-bottom:6px">'
                f'<span class="section-label">DFD Completeness</span>'
                f'<span style="font-size:0.85rem;font-weight:700;color:{cc}">{pct}%</span></div>'
                f'<div style="background:rgba(255,255,255,0.06);border-radius:6px;height:8px">'
                f'<div style="background:{cc};width:{pct}%;height:100%;border-radius:6px"></div></div></div>', unsafe_allow_html=True)
            st.markdown("**Risk Level**")
            rscore = {"Critical":1.0,"High":0.75,"Medium":0.5,"Low":0.25}.get(risk,0.5)
            st.plotly_chart(risk_gauge(risk, rscore), use_container_width=True, config={"displayModeBar": False})

        with threat_col:
            st.markdown(f"**Threats Identified ({len(threats)})**")
            if threats:
                for i, t in enumerate(threats):
                    cat = t.get("stride_category","Unknown"); conf = t.get("confidence","Low")
                    col_t = STRIDE_COLORS.get(cat,"#888"); cc2 = CONF_COLORS.get(conf,"#888")
                    icon = STRIDE_ICONS.get(cat,"🔒"); tid = t.get("threat_id",f"T{i+1}")
                    comp = t.get("affected_component",""); desc = t.get("threat_description","")
                    ctrl = t.get("missing_control","")
                    expl = t.get("explanation","") if st.session_state.get("show_detail", True) else ""
                    st.markdown(
                        f'<div class="card card-accent" style="border-left-color:{col_t}">'
                        f'<div style="display:flex;align-items:center;gap:8px;margin-bottom:8px">'
                        f'<span>{icon}</span>'
                        f'<span style="font-size:0.72rem;font-weight:700;color:#4a5568;font-family:monospace">{tid}</span>'
                        f'<span class="badge" style="background:{col_t}22;color:{col_t};border:1px solid {col_t}44">{cat}</span>'
                        f'<span style="margin-left:auto;font-size:0.72rem;font-weight:600;color:{cc2}">'
                        f'<span style="width:7px;height:7px;border-radius:50%;background:{cc2};display:inline-block;margin-right:4px"></span>{conf}</span></div>'
                        f'<div style="font-size:0.75rem;color:#4a9ede;font-family:monospace;margin-bottom:6px">{comp[:70]}</div>'
                        f'<div style="font-size:0.87rem;color:#9aa5b4;line-height:1.5;margin-bottom:6px">{desc}</div>'
                        f'<div style="font-size:0.82rem;background:rgba(255,255,255,0.04);border:1px solid rgba(255,255,255,0.07);border-radius:8px;padding:8px 12px;color:#63b3ed">🛡 {ctrl}</div>'
                        + (f'<div style="font-size:0.8rem;color:#718096;font-style:italic;margin-top:6px">💡 {expl}</div>' if expl else '') +
                        '</div>', unsafe_allow_html=True)
            else:
                st.info("No threats identified.")

        st.success("✅ Navigate to **📊 Threat Intelligence Dashboard** for detailed analysis views.")

# ═══════════════════════════════════════════════════════════════
# PAGE: THREAT INTELLIGENCE DASHBOARD
# ═══════════════════════════════════════════════════════════════
elif page == "📊 Threat Intelligence Dashboard":
    st.header("📊 Threat Intelligence Dashboard")

    if not st.session_state.analysis_result:
        st.warning("⚠️ No analysis available. Go to **📥 DFD Upload & Validation** to analyze a DFD first.")
        st.stop()

    result = st.session_state.analysis_result
    threats = result.get("threats", [])
    cov = result.get("stride_coverage", {})

    sub = st.radio("Module", ["STRIDE Overview","Node-Level Threat Mapping","Risk Heatmap","Severity Breakdown","Timeline View"],
                   horizontal=True)

    if not threats:
        st.info("No threats detected in the analysis.")
        st.stop()

    df = pd.DataFrame(threats)

    # ── STRIDE Overview ───────────────────────────────────────
    if sub == "STRIDE Overview":
        st.subheader("STRIDE Category Distribution")
        col1, col2 = st.columns([1,1], gap="large")
        with col1:
            stride_counts = df["stride_category"].value_counts().reset_index()
            stride_counts.columns = ["Category", "Count"]
            colors = [STRIDE_COLORS.get(c, "#888") for c in stride_counts["Category"]]
            fig = go.Figure(go.Bar(x=stride_counts["Category"], y=stride_counts["Count"],
                marker_color=colors, text=stride_counts["Count"], textposition='auto'))
            fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
                font=dict(color="#9aa5b4", family="Inter"), height=350,
                xaxis=dict(gridcolor="rgba(255,255,255,0.05)"),
                yaxis=dict(gridcolor="rgba(255,255,255,0.05)"))
            st.plotly_chart(fig, use_container_width=True, config={"displayModeBar": False})
        with col2:
            st.plotly_chart(radar_chart(cov), use_container_width=True, config={"displayModeBar": False})

        st.markdown("**Per-Category Breakdown**")
        for cat in STRIDE_COLORS:
            cat_threats = [t for t in threats if t.get("stride_category") == cat]
            if cat_threats:
                icon = STRIDE_ICONS.get(cat, "🔒")
                col_c = STRIDE_COLORS[cat]
                with st.expander(f"{icon} {cat} — {len(cat_threats)} threat(s)"):
                    for t in cat_threats:
                        st.markdown(f"- **{t.get('affected_component','')}**: {t.get('threat_description','')}")

    # ── Node-Level Threat Mapping ─────────────────────────────
    elif sub == "Node-Level Threat Mapping":
        st.subheader("Node-Level Threat Mapping")
        comp_col = "affected_component" if "affected_component" in df.columns else None
        if comp_col:
            nodes = df[comp_col].dropna().unique()
            selected = st.selectbox("Select Component", nodes)
            filtered = df[df[comp_col] == selected]
            st.markdown(f"**Threats targeting: `{selected}`**")
            for _, t in filtered.iterrows():
                cat = t.get("stride_category","Unknown")
                styled_card(
                    f'{STRIDE_ICONS.get(cat,"🔒")} <span class="badge" style="background:{STRIDE_COLORS.get(cat,"#888")}22;'
                    f'color:{STRIDE_COLORS.get(cat,"#888")}">{cat}</span> '
                    f'<span style="color:#9aa5b4;margin-left:8px">{t.get("threat_description","")}</span><br>'
                    f'<span style="color:#63b3ed;font-size:0.82rem">🛡 {t.get("missing_control","")}</span>',
                    STRIDE_COLORS.get(cat, "#888"))

            st.markdown("**Threat Density by Component**")
            node_counts = df[comp_col].value_counts().reset_index()
            node_counts.columns = ["Component", "Threats"]
            fig = px.bar(node_counts, x="Threats", y="Component", orientation="h",
                         color="Threats", color_continuous_scale=["#1a365d","#fc8181"])
            fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
                font=dict(color="#9aa5b4"), height=max(250, len(nodes)*40),
                yaxis=dict(gridcolor="rgba(255,255,255,0.05)"))
            st.plotly_chart(fig, use_container_width=True, config={"displayModeBar": False})
        else:
            st.info("No component data available.")

    # ── Risk Heatmap ──────────────────────────────────────────
    elif sub == "Risk Heatmap":
        st.subheader("Risk Heatmap: STRIDE × Components")
        comp_col = "affected_component" if "affected_component" in df.columns else None
        if comp_col:
            components = df[comp_col].dropna().unique().tolist()
            categories = list(STRIDE_COLORS.keys())
            matrix = []
            for comp in components:
                row = []
                for cat in categories:
                    count = len(df[(df[comp_col]==comp) & (df["stride_category"]==cat)])
                    row.append(count)
                matrix.append(row)
            fig = go.Figure(go.Heatmap(z=matrix, x=categories, y=components,
                colorscale=[[0,"#0d1b2a"],[0.5,"#d97706"],[1,"#dc2626"]],
                text=matrix, texttemplate="%{text}", textfont=dict(size=12, color="white"),
                hovertemplate="<b>%{y}</b><br>%{x}: %{z} threats<extra></extra>"))
            fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
                font=dict(color="#9aa5b4", family="Inter"), height=max(300, len(components)*50),
                xaxis=dict(side="top"), margin=dict(l=10,r=10,t=40,b=10))
            st.plotly_chart(fig, use_container_width=True, config={"displayModeBar": False})
        else:
            st.info("No component data available.")

    # ── Severity Breakdown ────────────────────────────────────
    elif sub == "Severity Breakdown":
        st.subheader("Severity Breakdown")
        conf_col = "confidence" if "confidence" in df.columns else None
        if conf_col:
            sev_counts = df[conf_col].value_counts().reset_index()
            sev_counts.columns = ["Confidence", "Count"]
            colors = [CONF_COLORS.get(c, "#888") for c in sev_counts["Confidence"]]
            fig = go.Figure(go.Pie(labels=sev_counts["Confidence"], values=sev_counts["Count"],
                marker=dict(colors=colors), hole=0.4, textinfo="label+percent",
                textfont=dict(color="white", size=12)))
            fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", font=dict(color="#9aa5b4"), height=350,
                              showlegend=True, legend=dict(font=dict(color="#9aa5b4")))
            st.plotly_chart(fig, use_container_width=True, config={"displayModeBar": False})

            st.markdown("**Threats by Confidence Level**")
            for conf_level in ["High", "Medium", "Low"]:
                conf_threats = [t for t in threats if t.get("confidence") == conf_level]
                if conf_threats:
                    cc = CONF_COLORS.get(conf_level, "#888")
                    with st.expander(f"{'🔴' if conf_level=='High' else '🟡' if conf_level=='Medium' else '🔵'} {conf_level} Confidence — {len(conf_threats)} threat(s)"):
                        for t in conf_threats:
                            st.markdown(f"- **[{t.get('stride_category','')}]** {t.get('affected_component','')}: {t.get('threat_description','')}")
        else:
            st.info("No confidence data available.")

    # ── Timeline View ─────────────────────────────────────────
    elif sub == "Timeline View":
        st.subheader("Analysis Timeline")
        ts = result.get("analysis_timestamp", datetime.now().isoformat())
        dur = result.get("analysis_duration_seconds", 0)
        risk = result.get("overall_risk_level", "Unknown")

        col1, col2, col3 = st.columns(3)
        with col1: metric_card(ts[:19], "Analysis Timestamp", "#63b3ed")
        with col2: metric_card(f"{dur}s", "Duration", "#68d391")
        with col3: metric_card(risk, "Risk Level", RISK_COLORS.get(risk, "#4a5568"))

        df["order"] = range(1, len(df)+1)
        fig = px.scatter(df, x="order", y="stride_category", size=[12]*len(df),
            color="stride_category", color_discrete_map=STRIDE_COLORS,
            hover_data=["affected_component","confidence"] if "affected_component" in df.columns else ["stride_category"])
        fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
            font=dict(color="#9aa5b4"), height=350, xaxis_title="Threat Discovery Order",
            yaxis_title="STRIDE Category", showlegend=False,
            xaxis=dict(gridcolor="rgba(255,255,255,0.05)"),
            yaxis=dict(gridcolor="rgba(255,255,255,0.05)"))
        st.plotly_chart(fig, use_container_width=True, config={"displayModeBar": False})


# ═══════════════════════════════════════════════════════════════
# PAGE: SECURITY ARCHITECTURE
# ═══════════════════════════════════════════════════════════════
elif page == "🛡 Security Architecture":
    st.header("🛡 Security Architecture")

    sub = st.radio("Module", ["Zero Trust Architecture","Defense-in-Depth Layers","Trust Boundary Analysis","Secure Design Principles"],
                   horizontal=True)

    if sub == "Zero Trust Architecture":
        st.subheader("Zero Trust Architecture")
        cols = st.columns(3, gap="large")
        for col, title, desc, icon in [
            (cols[0], "Never Trust", "All traffic is treated as potentially hostile regardless of source.", "🚫"),
            (cols[1], "Always Verify", "Every request must be authenticated, authorized, and encrypted.", "✅"),
            (cols[2], "Assume Breach", "Design systems assuming attackers are already inside the perimeter.", "🔓")]:
            with col:
                styled_card(f'<div style="font-size:1.8rem;margin-bottom:8px">{icon}</div>'
                    f'<div style="font-size:0.95rem;font-weight:700;color:#e2e8f0;margin-bottom:6px">{title}</div>'
                    f'<div style="font-size:0.82rem;color:#718096;line-height:1.6">{desc}</div>', "#63b3ed")

        st.markdown("""
        **Core Principles:**
        - **Identity-centric security** — Authenticate every user, device, and service
        - **Micro-segmentation** — Isolate workloads to limit lateral movement
        - **Least-privilege access** — Grant minimum permissions required
        - **Continuous monitoring** — Log and analyze all activity in real-time
        - **Dynamic policy enforcement** — Adapt access decisions based on risk signals
        """)

    elif sub == "Defense-in-Depth Layers":
        st.subheader("Defense-in-Depth Layers")
        layers = [
            ("🌐", "Perimeter", "WAF, DDoS protection, DNS filtering", "#fc8181"),
            ("🔒", "Network", "Firewalls, VPN, network segmentation, IDS/IPS", "#f6ad55"),
            ("🖥", "Host", "OS hardening, patch management, EDR, anti-malware", "#68d391"),
            ("⚙️", "Application", "Input validation, SAST/DAST, CSRF/XSS protection", "#63b3ed"),
            ("💾", "Data", "Encryption at rest/transit, tokenization, DLP, backup", "#b794f4"),
            ("👤", "Identity", "MFA, SSO, RBAC/ABAC, privileged access management", "#f687b3"),
        ]
        for icon, layer, desc, color in layers:
            styled_card(
                f'<div style="display:flex;align-items:center;gap:16px">'
                f'<span style="font-size:1.5rem">{icon}</span>'
                f'<div><div style="font-size:0.95rem;font-weight:700;color:#e2e8f0">{layer} Layer</div>'
                f'<div style="font-size:0.82rem;color:#718096">{desc}</div></div></div>', color)

    elif sub == "Trust Boundary Analysis":
        st.subheader("Trust Boundary Analysis")
        st.markdown("""
        Trust boundaries in DFDs define **security zones** where different levels of trust apply.
        Threats emerge most frequently at **boundary crossings** — where data moves between zones.

        **Common Trust Boundaries:**
        - **Internet ↔ DMZ** — External users accessing public-facing services
        - **DMZ ↔ Internal** — API gateways forwarding to backend services
        - **Internal ↔ Database** — Application servers accessing data stores
        - **Service ↔ Service** — Microservice-to-microservice communication
        """)
        if st.session_state.analysis_result:
            result = st.session_state.analysis_result
            crossings = [t for t in result.get("threats",[]) if "boundary" in t.get("explanation","").lower() or "trust" in t.get("explanation","").lower()]
            if crossings:
                st.markdown(f"**{len(crossings)} threats identified at boundary crossings:**")
                for t in crossings:
                    styled_card(
                        f'{STRIDE_ICONS.get(t["stride_category"],"🔒")} **[{t["stride_category"]}]** {t.get("affected_component","")}<br>'
                        f'<span style="color:#9aa5b4">{t.get("threat_description","")}</span>', "#f6ad55")
            else:
                st.info("Run an analysis to see boundary-crossing threats.")
        else:
            st.info("Run an analysis to see boundary-crossing threats here.")

    elif sub == "Secure Design Principles":
        st.subheader("Secure Design Principles")
        principles = [
            ("Least Privilege", "Grant only the minimum access level required for each operation."),
            ("Fail Securely", "System failures should default to a secure state, not an open one."),
            ("Defense in Depth", "Use multiple layers of security controls — no single point of failure."),
            ("Separation of Duties", "Critical operations require multiple parties to prevent insider abuse."),
            ("Economy of Mechanism", "Keep security mechanisms simple — complexity breeds vulnerabilities."),
            ("Complete Mediation", "Every access to every resource must be validated against the access control system."),
            ("Open Design", "Security should not depend on secrecy of the design (Kerckhoffs' principle)."),
            ("Psychological Acceptability", "Security mechanisms must not make the system harder to use than without them."),
        ]
        for title, desc in principles:
            styled_card(f'<div style="font-size:0.95rem;font-weight:700;color:#e2e8f0;margin-bottom:4px">{title}</div>'
                f'<div style="font-size:0.82rem;color:#718096;line-height:1.5">{desc}</div>', "#3182ce")


# ═══════════════════════════════════════════════════════════════
# PAGE: STRIDE METHODOLOGY
# ═══════════════════════════════════════════════════════════════
elif page == "📚 STRIDE Methodology":
    st.header("📚 STRIDE Methodology")

    sub = st.radio("Module", ["Theory of STRIDE","Mapping to DFD Elements","Academic References","Model Limitations"],
                   horizontal=True)

    if sub == "Theory of STRIDE":
        st.subheader("STRIDE Threat Classification Framework")
        st.markdown("Developed by Microsoft, STRIDE categorizes threats into six classes:")
        stride_info = [
            ("Spoofing", "Illegitimately assuming another identity.", "Authentication mechanisms, digital signatures, MFA", "#fc8181"),
            ("Tampering", "Unauthorized modification of data or code.", "Integrity checks, digital signatures, input validation", "#f6ad55"),
            ("Repudiation", "Denying having performed an action without proof.", "Audit logging, digital signatures, timestamps, non-repudiation protocols", "#68d391"),
            ("Information Disclosure", "Exposing information to unauthorized parties.", "Encryption, access controls, data classification, DLP", "#63b3ed"),
            ("Denial of Service", "Making a system unavailable to legitimate users.", "Rate limiting, load balancing, redundancy, CDN", "#b794f4"),
            ("Elevation of Privilege", "Gaining capabilities beyond those authorized.", "RBAC, input validation, sandboxing, least privilege", "#f687b3"),
        ]
        for name, desc, mitigation, color in stride_info:
            icon = STRIDE_ICONS[name]
            styled_card(
                f'<div style="display:flex;align-items:flex-start;gap:12px">'
                f'<span style="font-size:1.5rem">{icon}</span>'
                f'<div><div style="font-size:1rem;font-weight:700;color:#e2e8f0;margin-bottom:4px">{name}</div>'
                f'<div style="font-size:0.85rem;color:#9aa5b4;margin-bottom:6px">{desc}</div>'
                f'<div style="font-size:0.78rem;color:#63b3ed">🛡 Mitigations: {mitigation}</div></div></div>', color)

    elif sub == "Mapping to DFD Elements":
        st.subheader("STRIDE ↔ DFD Element Mapping")
        mapping_data = {
            "DFD Element": ["External Entity","Process","Data Store","Data Flow","Data Flow","External Entity"],
            "STRIDE Category": ["Spoofing","Tampering","Information Disclosure","Denial of Service","Tampering","Repudiation"],
            "Example Threat": [
                "Attacker impersonates a legitimate user",
                "Malicious input modifies processing logic",
                "Unauthorized access to stored credentials",
                "Flooding a data flow to disrupt service",
                "MITM attack modifies data in transit",
                "User denies performing a transaction"
            ]}
        st.dataframe(pd.DataFrame(mapping_data), use_container_width=True, hide_index=True)

        st.markdown("""
        **Key Mapping Rules:**
        - **External Entities** → Spoofing, Repudiation
        - **Processes** → Tampering, Information Disclosure, Denial of Service, Elevation of Privilege
        - **Data Stores** → Tampering, Information Disclosure, Denial of Service
        - **Data Flows** → Tampering, Information Disclosure, Denial of Service
        """)

    elif sub == "Academic References":
        st.subheader("Academic References")
        refs = [
            ("Shostack, A. (2014)", "Threat Modeling: Designing for Security", "Wiley. The foundational text on STRIDE threat modeling methodology."),
            ("Howard, M. & Lipner, S. (2006)", "The Security Development Lifecycle", "Microsoft Press. SDL practices including STRIDE integration."),
            ("OWASP Foundation (2021)", "OWASP Threat Modeling Playbook", "Community-driven guide to practical threat modeling approaches."),
            ("Xiong, W. & Lagerström, R. (2019)", "Threat Modeling of Cloud Computing Systems", "IEEE. Adapting STRIDE for cloud-native architectures."),
            ("Dhillon, D. (2011)", "Developer-Driven Threat Modeling", "IEEE Security & Privacy. Integrating threat modeling into agile workflows."),
        ]
        for author, title, desc in refs:
            styled_card(f'<div style="font-size:0.82rem;color:#63b3ed;font-weight:600">{author}</div>'
                f'<div style="font-size:0.92rem;font-weight:700;color:#e2e8f0;margin:4px 0">{title}</div>'
                f'<div style="font-size:0.8rem;color:#718096">{desc}</div>', "#3182ce")

    elif sub == "Model Limitations":
        st.subheader("Model Limitations & Considerations")
        limitations = [
            ("LLM Hallucination Risk", "The model may generate plausible-sounding but incorrect threats. Always validate findings with domain experts.", "🤖"),
            ("Context Dependency", "Output quality depends heavily on the completeness and accuracy of the input DFD and security context.", "📋"),
            ("Structured Input Required", "The system requires JSON-formatted DFD representation — cannot process diagrams directly.", "📐"),
            ("No Code-Level Analysis", "STRIDE analysis operates at the architectural level, not at source code level.", "💻"),
            ("Temporal Limitations", "The LLM's knowledge has a training cutoff — may not include the latest CVEs or attack patterns.", "⏰"),
            ("Determinism", "Results may vary slightly between runs due to LLM temperature settings.", "🎲"),
        ]
        for title, desc, icon in limitations:
            styled_card(f'<div style="display:flex;gap:12px;align-items:flex-start">'
                f'<span style="font-size:1.3rem">{icon}</span>'
                f'<div><div style="font-size:0.92rem;font-weight:700;color:#fc8181;margin-bottom:4px">{title}</div>'
                f'<div style="font-size:0.82rem;color:#718096;line-height:1.5">{desc}</div></div></div>', "#fc8181")


# ═══════════════════════════════════════════════════════════════
# PAGE: COMPLIANCE MAPPING
# ═══════════════════════════════════════════════════════════════
elif page == "📜 Compliance Mapping":
    st.header("📜 Compliance Mapping")

    sub = st.radio("Module", ["OWASP Top 10","NIST 800-53","ISO 27001","GDPR Considerations"], horizontal=True)

    if sub == "OWASP Top 10":
        st.subheader("OWASP Top 10 (2021) ↔ STRIDE Mapping")
        owasp_data = [
            ("A01", "Broken Access Control", "Elevation of Privilege", "Enforce RBAC, deny by default, rate limit APIs"),
            ("A02", "Cryptographic Failures", "Information Disclosure", "Use strong algorithms (AES-256, RSA-2048), enforce TLS"),
            ("A03", "Injection", "Tampering", "Parameterized queries, input validation, WAF rules"),
            ("A04", "Insecure Design", "All STRIDE Categories", "Threat modeling (this tool!), secure design patterns"),
            ("A05", "Security Misconfiguration", "Information Disclosure", "Hardened configs, automated scanning, remove defaults"),
            ("A06", "Vulnerable Components", "Tampering", "SCA tools, dependency updates, SBOM tracking"),
            ("A07", "Auth & Session Failures", "Spoofing", "MFA, session timeout, secure token storage"),
            ("A08", "Software Integrity Failures", "Tampering", "Code signing, CI/CD pipeline security, SBOM"),
            ("A09", "Logging & Monitoring Failures", "Repudiation", "Centralized logging, SIEM, alerting, audit trails"),
            ("A10", "SSRF", "Information Disclosure", "Input validation, network segmentation, allowlists"),
        ]
        for code, name, stride, mitigation in owasp_data:
            col_c = STRIDE_COLORS.get(stride, "#888")
            styled_card(
                f'<div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:6px">'
                f'<span style="font-weight:700;color:#e2e8f0">{code}: {name}</span>'
                f'<span class="badge" style="background:{col_c}22;color:{col_c};border:1px solid {col_c}44">{stride}</span></div>'
                f'<div style="font-size:0.8rem;color:#63b3ed">🛡 {mitigation}</div>', col_c)

    elif sub == "NIST 800-53":
        st.subheader("NIST SP 800-53 Control Families ↔ STRIDE")
        nist_data = [
            ("AC", "Access Control", "Spoofing, Elevation of Privilege", "Account management, access enforcement, separation of duties"),
            ("AU", "Audit & Accountability", "Repudiation", "Audit events, content, storage, non-repudiation"),
            ("SC", "System & Comms Protection", "Information Disclosure, Tampering", "Boundary protection, cryptographic protection, transmission confidentiality"),
            ("IA", "Identification & Auth", "Spoofing", "Multi-factor auth, authenticator management, credential protection"),
            ("SI", "System & Info Integrity", "Tampering", "Flaw remediation, malicious code protection, input validation"),
            ("CP", "Contingency Planning", "Denial of Service", "Recovery plans, backup, system reconstitution"),
        ]
        for code, family, stride, controls in nist_data:
            styled_card(
                f'<div style="font-weight:700;color:#e2e8f0;margin-bottom:4px">{code} — {family}</div>'
                f'<div style="font-size:0.8rem;color:#b794f4;margin-bottom:4px">STRIDE: {stride}</div>'
                f'<div style="font-size:0.8rem;color:#718096">Controls: {controls}</div>', "#b794f4")

    elif sub == "ISO 27001":
        st.subheader("ISO 27001:2022 Controls ↔ STRIDE")
        iso_data = [
            ("A.5", "Organizational Controls", "Policies, roles, responsibilities, threat intelligence"),
            ("A.6", "People Controls", "Screening, awareness training, disciplinary process"),
            ("A.7", "Physical Controls", "Physical entry, securing offices, equipment protection"),
            ("A.8", "Technological Controls", "Endpoint security, access rights, cryptography, logging, network security"),
        ]
        for code, name, desc in iso_data:
            styled_card(f'<div style="font-weight:700;color:#e2e8f0">{code} — {name}</div>'
                f'<div style="font-size:0.82rem;color:#718096;margin-top:4px">{desc}</div>', "#68d391")

        st.markdown("""
        **ISO 27001 Risk Management Process:**
        1. **Context Establishment** — Define scope and risk criteria
        2. **Risk Identification** — Identify assets, threats, vulnerabilities *(SecureByDesign automates this)*
        3. **Risk Analysis** — Assess likelihood and impact
        4. **Risk Evaluation** — Compare against risk criteria
        5. **Risk Treatment** — Select and implement controls
        """)

    elif sub == "GDPR Considerations":
        st.subheader("GDPR ↔ Threat Modeling")
        gdpr_data = [
            ("Article 25", "Data Protection by Design", "Integrate privacy controls from system inception — SecureByDesign supports this.", "#63b3ed"),
            ("Article 32", "Security of Processing", "Implement appropriate technical and organizational measures.", "#68d391"),
            ("Article 33", "Breach Notification", "Report breaches within 72 hours — requires logging (Repudiation prevention).", "#f6ad55"),
            ("Article 35", "DPIA", "Conduct Data Protection Impact Assessment for high-risk processing.", "#b794f4"),
            ("Article 5(1)(f)", "Integrity & Confidentiality", "Protect against unauthorized access, loss, destruction (STRIDE coverage).", "#fc8181"),
        ]
        for article, title, desc, color in gdpr_data:
            styled_card(
                f'<div style="font-size:0.78rem;font-weight:700;color:{color}">{article}</div>'
                f'<div style="font-size:0.95rem;font-weight:700;color:#e2e8f0;margin:4px 0">{title}</div>'
                f'<div style="font-size:0.82rem;color:#718096;line-height:1.5">{desc}</div>', color)


# ═══════════════════════════════════════════════════════════════
# PAGE: MODEL CONFIGURATION
# ═══════════════════════════════════════════════════════════════
elif page == "⚙ Model Configuration":
    st.header("⚙ Model Configuration")

    st.markdown("**Current Pipeline Configuration**")
    col1, col2 = st.columns(2, gap="large")
    with col1:
        styled_card(
            '<div style="font-size:0.78rem;color:#4a5568;font-weight:600;text-transform:uppercase;letter-spacing:0.1em">LLM Backend</div>'
            '<div style="font-size:1.1rem;font-weight:700;color:#63b3ed;margin:8px 0">llama-3.3-70b-versatile</div>'
            '<div style="font-size:0.8rem;color:#718096">Provider: Groq (free tier) · Context: 128K tokens</div>', "#63b3ed")
        styled_card(
            '<div style="font-size:0.78rem;color:#4a5568;font-weight:600;text-transform:uppercase;letter-spacing:0.1em">Temperature</div>'
            '<div style="font-size:1.1rem;font-weight:700;color:#68d391;margin:8px 0">0.1</div>'
            '<div style="font-size:0.8rem;color:#718096">Low temperature for deterministic, consistent JSON output</div>', "#68d391")
    with col2:
        styled_card(
            '<div style="font-size:0.78rem;color:#4a5568;font-weight:600;text-transform:uppercase;letter-spacing:0.1em">Max Output Tokens</div>'
            '<div style="font-size:1.1rem;font-weight:700;color:#f6ad55;margin:8px 0">4,096</div>'
            '<div style="font-size:0.8rem;color:#718096">Sufficient for comprehensive STRIDE analysis with explanations</div>', "#f6ad55")
        styled_card(
            '<div style="font-size:0.78rem;color:#4a5568;font-weight:600;text-transform:uppercase;letter-spacing:0.1em">Retry Policy</div>'
            '<div style="font-size:1.1rem;font-weight:700;color:#b794f4;margin:8px 0">3 attempts · Linear backoff</div>'
            '<div style="font-size:0.8rem;color:#718096">Base delay: 2s · Handles Groq rate limits gracefully</div>', "#b794f4")

    st.markdown("**DFD Parser Capabilities**")
    caps = ["Accepts any JSON key format (name/label, from/source/sender, etc.)",
            "Auto-normalizes 150+ key aliases to canonical schema",
            "Handles camelCase, snake_case, and mixed formats",
            "Auto-generates missing IDs and DFD identifiers",
            "Fuzzy node type matching (actor→external_entity, database→datastore, etc.)"]
    for c in caps:
        st.markdown(f'<div style="padding:4px 0;font-size:0.85rem;color:#9aa5b4">✅ {c}</div>', unsafe_allow_html=True)


# ═══════════════════════════════════════════════════════════════
# PAGE: REPORT GENERATION
# ═══════════════════════════════════════════════════════════════
elif page == "📄 Report Generation":
    st.header("📄 Report Generation")

    if not st.session_state.analysis_result:
        st.warning("⚠️ No analysis available. Go to **📥 DFD Upload & Validation** to analyze a DFD first.")
    else:
        result = st.session_state.analysis_result
        threats = result.get("threats", [])

        col1, col2, col3 = st.columns(3)
        with col1: metric_card(str(len(threats)), "Total Threats", "#fc8181")
        with col2: metric_card(result.get("overall_risk_level","?"), "Risk Level", RISK_COLORS.get(result.get("overall_risk_level",""),"#4a5568"))
        with col3: metric_card(f'{result.get("analysis_duration_seconds",0)}s', "Duration", "#63b3ed")

        st.markdown("---")

        mc = result.get("missing_controls_summary", [])
        if mc:
            st.markdown("**Missing Security Controls**")
            for i, c in enumerate(mc):
                styled_card(
                    f'<span style="font-size:0.72rem;font-weight:800;color:#3182ce;background:rgba(49,130,206,0.12);'
                    f'border:1px solid rgba(49,130,206,0.25);border-radius:6px;padding:2px 8px">MC-{i+1}</span> '
                    f'<span style="font-size:0.85rem;color:#9aa5b4;margin-left:8px">{c}</span>', "#3182ce")

        st.markdown("---")
        dl_col, raw_col = st.columns(2)
        with dl_col:
            st.download_button("⬇️ Download Threat Report (JSON)",
                data=json.dumps(result, indent=2),
                file_name=f"threat_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json",
                mime="application/json", use_container_width=True)
        with raw_col:
            with st.expander("📋 Raw JSON Output"):
                st.json(result)


In [ ]:
print("✅ Streamlit app written to /kaggle/working/SecureByDesign/app/streamlit_app.py")

## Phase 3 — Evaluation Harness
Run the AI pipeline on 17 real microservice DFDs from the **microSecEnD** dataset and compute Precision, Recall, and F1-Score against STRIDE ground truth.

> **Requires:** `GROQ_API_KEY` in Kaggle Secrets — get free key at [console.groq.com/keys](https://console.groq.com/keys).

**Ground truth derivation:** STRIDE categories are derived from security stereotypes in microSecEnD:
- `plaintext_credentials` → Information Disclosure + Tampering
- `csrf_disabled` → Tampering
- Missing `authentication` on flows → Spoofing
- Missing `local_logging` → Repudiation
- `entrypoint`/`gateway` without rate limiting → Denial of Service
- Missing `authorization` on business services → Elevation of Privilege

In [ ]:
# ── Step 3.1: Build Test Suite from microSecEnD ──────────────────────────
import json, os, sys

WORK = '/kaggle/working/SecureByDesign'
EVAL_DIR = f'{WORK}/evaluation'
TEST_DFD_DIR = f'{EVAL_DIR}/test_dfds'
MICROSECEND = f'{WORK}/data/microSecEnD/dataset'
os.makedirs(TEST_DFD_DIR, exist_ok=True)

STRIDE_CATS = ['Spoofing','Tampering','Repudiation','Information Disclosure','Denial of Service','Elevation of Privilege']

SVC_ST = {'plaintext_credentials': ['Information Disclosure','Tampering'], 'csrf_disabled': ['Tampering']}
FLOW_ST = {'plaintext_credentials_link': ['Information Disclosure','Tampering'],
           'authentication_with_plaintext_credentials': ['Information Disclosure','Spoofing']}

def derive_gt(services, flows, ext):
    s = set()
    for svc in services:
        for st in svc.get('stereotypes', []):
            for c in SVC_ST.get(st, []): s.add(c)
    for fl in flows:
        for st in fl.get('stereotypes', []):
            for c in FLOW_ST.get(st, []): s.add(c)
        if 'authenticated' not in fl.get('stereotypes', []): s.add('Spoofing')
    biz = [sv for sv in services if 'infrastructural' not in sv.get('stereotypes',[]) and 'database' not in sv.get('stereotypes',[])]
    for sv in biz:
        if 'local_logging' not in sv.get('stereotypes',[]): s.add('Repudiation'); break
    for sv in services:
        if 'entrypoint' in sv.get('stereotypes',[]) or 'gateway' in sv.get('stereotypes',[]): s.add('Denial of Service'); break
    for sv in biz:
        if 'authorization' not in sv.get('stereotypes',[]): s.add('Elevation of Privilege'); break
    return sorted(s)

def convert(name, data):
    svcs = data.get('services',[]); flows = data.get('information_flows',[]); exts = data.get('external_entities',[])
    nodes, nmap = [], {}
    for i, sv in enumerate(svcs):
        nid = f'S{i+1}'; nmap[sv['name']] = nid
        nodes.append({'id':nid, 'type':'datastore' if 'database' in sv.get('stereotypes',[]) else 'process', 'name':sv['name']})
    for i, e in enumerate(exts):
        nid = f'EXT{i+1}'; nmap[e['name']] = nid
        nodes.append({'id':nid, 'type':'external_entity', 'name':e['name']})
    edges = []
    for i, fl in enumerate(flows):
        if fl['sender'] not in nmap or fl['receiver'] not in nmap: continue
        st = fl.get('stereotypes',[])
        edges.append({'id':f'F{i+1}','from':nmap[fl['sender']],'to':nmap[fl['receiver']],
            'data_description':f"{fl['sender']} -> {fl['receiver']}",
            'protocol':'JDBC' if 'jdbc' in st else 'HTTP/REST' if 'restful_http' in st else None,
            'authenticated': True if 'authenticated' in st else None,
            'encrypted': False if 'plaintext_credentials_link' in st else None})
    tb = []
    gw = [n for n in nodes if any(s in next((sv.get('stereotypes',[]) for sv in svcs if sv['name']==n['name']),[]) for s in ['gateway','entrypoint'])]
    en = [n for n in nodes if n['type']=='external_entity']
    if gw and en: tb.append({'id':'TB1','name':'Internet Boundary','separates':[en[0]['id'],gw[0]['id']]})
    return {'dfd_id':name,'system_name':name.replace('_',' ').title(),'nodes':nodes,'edges':edges,'trust_boundaries':tb,
            'partial_info_flags':{'missing_trust_boundaries':len(tb)==0,'unknown_protocols':any(e['protocol'] is None for e in edges),
                                  'unspecified_auth':any(e['authenticated'] is None for e in edges),'incomplete_nodes':False}}

ground_truth = {}
count = 0
for pdir in sorted(os.listdir(MICROSECEND)):
    ppath = os.path.join(MICROSECEND, pdir)
    if not os.path.isdir(ppath): continue
    jsons = [f for f in os.listdir(ppath) if f.endswith('.json') and 'traceability' not in f and 'rules' not in f]
    if not jsons: continue
    with open(os.path.join(ppath, jsons[0]), 'r') as f:
        try: data = json.load(f)
        except: continue
    if 'services' not in data: continue
    dfd = convert(pdir, data)
    with open(f'{TEST_DFD_DIR}/{pdir}.json','w') as f: json.dump(dfd, f, indent=2)
    svcs = data.get('services',[]); flows = data.get('information_flows',[]); exts = data.get('external_entities',[])
    gt = derive_gt(svcs, flows, exts)
    ground_truth[pdir] = {'system_name':dfd['system_name'],'expected_stride_categories':gt,
                          'num_nodes':len(dfd['nodes']),'num_edges':len(dfd['edges'])}
    count += 1
    print(f'  [{count}] {pdir}: {len(dfd["nodes"])} nodes, {len(dfd["edges"])} edges, STRIDE={gt}')

with open(f'{EVAL_DIR}/ground_truth.json','w') as f: json.dump(ground_truth, f, indent=2)
print(f'\n\u2705 Test suite built: {count} DFDs with ground truth')

In [ ]:
# ── Step 3.2: Run Full Evaluation ────────────────────────────────────────
# This calls analyze_dfd() for each of the 17 test DFDs and computes metrics.
# NOTE: Takes ~2-5 min depending on Groq rate limits.

import time
from datetime import datetime
sys.path.insert(0, WORK)
from pipeline.inference import analyze_dfd

with open(f'{EVAL_DIR}/ground_truth.json') as f:
    ground_truth = json.load(f)

results = []
print(f'\n{"="*60}')
print('SECUREBYDESIGN EVALUATION RUN')
print(f'Timestamp: {datetime.now().isoformat()}')
print(f'{"="*60}\n')

test_files = sorted([f for f in os.listdir(TEST_DFD_DIR) if f.endswith('.json')])
for i, fname in enumerate(test_files):
    dfd_id = fname.replace('.json','')
    if dfd_id not in ground_truth: continue
    with open(f'{TEST_DFD_DIR}/{fname}') as f: dfd_json = json.load(f)
    gt = ground_truth[dfd_id]
    t0 = time.time()
    try:
        result = analyze_dfd(dfd_json, '')
        dur = time.time() - t0
        err = result.get('error')
    except Exception as e:
        dur = time.time() - t0
        results.append({'dfd_id':dfd_id,'precision':0,'recall':0,'f1':0,'threats':0,'error':str(e)})
        print(f'  [{i+1}/{len(test_files)}] {dfd_id}: ERROR - {e}')
        time.sleep(2); continue
    pred = list(set(t.get('stride_category') for t in result.get('threats',[]) if t.get('stride_category') in STRIDE_CATS))
    exp = gt.get('expected_stride_categories',[])
    ev, pv = [1 if c in exp else 0 for c in STRIDE_CATS], [1 if c in pred else 0 for c in STRIDE_CATS]
    tp = sum(e==1 and p==1 for e,p in zip(ev,pv))
    fp = sum(e==0 and p==1 for e,p in zip(ev,pv))
    fn = sum(e==1 and p==0 for e,p in zip(ev,pv))
    prec = tp/(tp+fp) if tp+fp else 0; rec = tp/(tp+fn) if tp+fn else 0
    f1 = 2*prec*rec/(prec+rec) if prec+rec else 0
    results.append({'dfd_id':dfd_id,'precision':round(prec,3),'recall':round(rec,3),'f1':round(f1,3),
                    'threats':len(result.get('threats',[])), 'risk':result.get('overall_risk_level','?'),
                    'duration':round(dur,1),'error':err,'predicted':pred,'expected':exp})
    print(f'  [{i+1}/{len(test_files)}] {dfd_id}: P={prec:.3f} R={rec:.3f} F1={f1:.3f} threats={len(result.get("threats",[]))}')
    time.sleep(2)  # rate limit

# ── Aggregate Results ──────────────────────────────────────────────────────
valid = [r for r in results if r.get('error') is None]
if valid:
    avg_p = sum(r['precision'] for r in valid)/len(valid)
    avg_r = sum(r['recall'] for r in valid)/len(valid)
    avg_f1 = sum(r['f1'] for r in valid)/len(valid)
else:
    avg_p = avg_r = avg_f1 = 0

print(f'\n{"="*60}')
print(f'AGGREGATE RESULTS ({len(valid)}/{len(results)} successful)')
print(f'{"-"*60}')
print(f'  Macro Precision : {avg_p:.3f}')
print(f'  Macro Recall    : {avg_r:.3f}')
print(f'  Macro F1-Score  : {avg_f1:.3f}')
print(f'{"="*60}')

# Save results
os.makedirs(f'{EVAL_DIR}/results', exist_ok=True)
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
rpath = f'{EVAL_DIR}/results/evaluation_{ts}.json'
with open(rpath, 'w') as f:
    json.dump({'aggregate':{'precision':round(avg_p,3),'recall':round(avg_r,3),'f1':round(avg_f1,3)},
               'per_dfd':results}, f, indent=2, default=str)
print(f'\nResults saved: {rpath}')

In [ ]:
# ── Step 3.3: Display Results Table ──────────────────────────────────────
import pandas as pd

df = pd.DataFrame(results)[['dfd_id','precision','recall','f1','threats','error']]
df = df.rename(columns={'dfd_id':'DFD','precision':'Precision','recall':'Recall','f1':'F1','threats':'Threats'})
display(df.style.format({'Precision':'{:.3f}','Recall':'{:.3f}','F1':'{:.3f}'}).background_gradient(subset=['F1'], cmap='RdYlGn'))

print(f'\nMacro Average \u2014 Precision: {avg_p:.3f} | Recall: {avg_r:.3f} | F1: {avg_f1:.3f}')

## Phase 5 — Launch Demo via ngrok
> **Requires:** `NGROK_TOKEN` in Kaggle Secrets — get free token at [ngrok.com](https://ngrok.com).

In [ ]:
!pip install pyngrok --quiet

from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient
import subprocess, time

try:
    ngrok_token = UserSecretsClient().get_secret("NGROK_TOKEN")
    ngrok.set_auth_token(ngrok_token)
    print("✅ Ngrok token set from Kaggle Secrets")
except Exception as e:
    print(f"⚠️  Could not get NGROK_TOKEN: {e}")
    print("   Add it via Add-ons → Secrets → NGROK_TOKEN")

ngrok.kill()

proc = subprocess.Popen([
    "streamlit", "run", "/kaggle/working/SecureByDesign/app/streamlit_app.py",
    "--server.port", "8501",
    "--server.headless", "true",
    "--browser.gatherUsageStats", "false"
])
time.sleep(6)

try:
    url = ngrok.connect(8501)
    print("\n" + "="*60)
    print(f"🔐 SecureByDesign Public Demo URL: {url}")
    print("="*60)
    print("Share this URL with your supervisor. Keep this cell running!")
except Exception as e:
    print(f"❌ Failed to create tunnel: {e}")